# Part 7A - Paged KV Cache: Block Allocation & Memory Management

In the previous parts, we built a continuous batching scheduler with token-budget and KV-capacity-aware scheduling. Our scheduler could estimate how much KV cache each request needed, decide whether a request could be admitted, and block execution when memory capacity was insufficient.

However, we treated KV memory as a simple capacity counter.

Real inference engines need to solve a more difficult problem: **how do we physically allocate, track, grow, and release KV cache memory for multiple requests with unpredictable sequence lengths?**

This is where **Paged KV Cache Management** comes in.

Inspired by the PagedAttention design introduced in [vLLM](https://github.com/vllm-project/vllm), we'll build a simplified block-based KV cache manager and integrate it with our existing scheduling concepts.

## What We'll Cover

1. **Why Contiguous KV Allocation Is Inefficient**
   - Memory fragmentation and wasted KV capacity
   - Why reserving memory for maximum sequence lengths is inefficient
   - How dynamic sequence growth complicates allocation

2. **Fixed-Size KV Blocks**
   - Dividing KV cache memory into fixed-size physical blocks
   - Calculating KV memory consumption per block
   - Understanding logical versus physical blocks

3. **Physical Block Pool & Free-List Allocator**
   - Managing a finite pool of KV blocks
   - Allocating blocks when requests need more capacity
   - Releasing blocks when requests finish

4. **Per-Request Block Tables**
   - Mapping logical blocks to physical GPU memory blocks
   - Tracking KV allocation independently for each request
   - Supporting noncontiguous physical memory allocation

5. **Dynamic KV Cache Growth**
   - Allocating additional blocks as sequences generate tokens
   - Reusing available space within the final allocated block
   - Handling requests that cannot obtain additional blocks

6. **Integrating Block Allocation With Our Scheduler**
   - Replacing token-level KV capacity accounting with block-level accounting
   - Checking memory availability before prefill and decode
   - Supporting multiple requests competing for KV blocks
   - Releasing blocks when requests finish

## What We'll Build

By the end of this notebook, we'll have a working **block-based KV cache manager** that:

- Maintains a finite pool of physical KV blocks.
- Allocates blocks dynamically to individual requests.
- Maintains logical-to-physical block mappings.
- Expands allocations as sequences grow.
- Reclaims KV blocks when requests complete.
- Works with a capacity-aware inference scheduler.

We'll focus on the allocation and scheduling mechanisms used in real inference systems, while simplifying the implementation enough to run in Google Colab.

**Scope:** This notebook implements KV block management and scheduling, not the actual PagedAttention CUDA kernel. We'll also distinguish block allocation from writing real K/V tensors into GPU memory.

In **Part 7B — PagedAttention**, we'll use these block mappings to understand how attention kernels access physically noncontiguous KV cache memory and verify attention computations against a contiguous reference implementation.

---

**The central idea:** A request's KV cache does not need to occupy one contiguous region of GPU memory. By splitting KV storage into fixed-size blocks and maintaining a mapping between logical and physical blocks, the inference engine can allocate memory incrementally and reuse freed capacity across requests.

In [2]:
!pip -q install transformers


In [3]:

from transformers import AutoConfig
import math

MODEL_ID = "Qwen/Qwen2.5-0.5B"

config = AutoConfig.from_pretrained(MODEL_ID)

num_layers = config.num_hidden_layers
num_attention_heads = config.num_attention_heads
num_kv_heads = config.num_key_value_heads
head_dim = getattr(
    config,
    "head_dim",
    config.hidden_size // num_attention_heads
)

print("Model:", MODEL_ID)
print("Transformer layers:", num_layers)
print("Attention heads:", num_attention_heads)
print("KV heads:", num_kv_heads)
print("Head dimension:", head_dim)


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-0.5B
Transformer layers: 24
Attention heads: 14
KV heads: 2
Head dimension: 64


In [4]:
#calculating kv memory per token


BYTES_PER_ELEMENT = 2  # FP16 KV cache

kv_bytes_per_token = (
    2
    * num_layers
    * num_kv_heads
    * head_dim
    * BYTES_PER_ELEMENT
)

print(f"KV bytes per token: {kv_bytes_per_token:,}")
print(f"KV KiB per token: {kv_bytes_per_token / 1024:.2f}")


KV bytes per token: 12,288
KV KiB per token: 12.00


In [5]:
#kv memory per block


BLOCK_SIZE = 16

kv_bytes_per_block = kv_bytes_per_token * BLOCK_SIZE

print("Tokens per block:", BLOCK_SIZE)
print(f"KV memory per block: {kv_bytes_per_block:,} bytes")
print(f"KV memory per block: {kv_bytes_per_block / 1024:.2f} KiB")


Tokens per block: 16
KV memory per block: 196,608 bytes
KV memory per block: 192.00 KiB


In [6]:
#calculating how many blocks a hypothetical request needs



def blocks_required(num_tokens: int) -> int:
    if num_tokens < 0:
        raise ValueError("Token count cannot be negative")

    return (num_tokens + BLOCK_SIZE - 1) // BLOCK_SIZE


request_lengths = {
    "A": 40,
    "B": 43,
    "C": 38,
}

for request_id, num_tokens in request_lengths.items():
    blocks = blocks_required(num_tokens)
    allocated_tokens = blocks * BLOCK_SIZE
    unused_slots = allocated_tokens - num_tokens

    print(
        f"Request {request_id}: "
        f"{num_tokens} tokens | "
        f"{blocks} blocks | "
        f"{unused_slots} unused token slots"
    )


Request A: 40 tokens | 3 blocks | 8 unused token slots
Request B: 43 tokens | 3 blocks | 5 unused token slots
Request C: 38 tokens | 3 blocks | 10 unused token slots


In [7]:
#memory accounting


total_used_tokens = sum(request_lengths.values())

total_blocks = sum(
    blocks_required(n)
    for n in request_lengths.values()
)

total_allocated_slots = total_blocks * BLOCK_SIZE
unused_slots = total_allocated_slots - total_used_tokens

total_allocated_bytes = total_blocks * kv_bytes_per_block
useful_bytes = total_used_tokens * kv_bytes_per_token

print(f"Actual tokens: {total_used_tokens}")
print(f"Allocated blocks: {total_blocks}")
print(f"Allocated token slots: {total_allocated_slots}")
print(f"Unused token slots: {unused_slots}")
print(f"Allocated KV memory: {total_allocated_bytes / 1024:.2f} KiB")
print(f"Useful KV storage: {useful_bytes / 1024:.2f} KiB")
print(f"Block utilization: {100 * total_used_tokens / total_allocated_slots:.2f}%")


Actual tokens: 121
Allocated blocks: 9
Allocated token slots: 144
Unused token slots: 23
Allocated KV memory: 1728.00 KiB
Useful KV storage: 1452.00 KiB
Block utilization: 84.03%


We'll start by understanding KV memory requirements, then progressively build a physical block allocator and integrate it with our existing scheduling concepts.

## 1. Model Configuration

We'll use **Qwen2.5-0.5B** to calculate realistic KV cache memory requirements.

We only load the model's configuration, not its weights, since this part focuses on memory management rather than running inference.

The relevant architecture parameters are:

```text
Model: Qwen/Qwen2.5-0.5B
Transformer layers: 24
Attention heads: 14
KV heads: 2
Head dimension: 64
```

Qwen2.5-0.5B uses **Grouped Query Attention (GQA)**, where multiple query heads share a smaller number of key/value heads.

This matters because KV cache memory scales with the number of KV heads, rather than the total number of query heads.

## 2. Calculating KV Cache Memory per Token

During autoregressive generation, the inference engine stores the Key and Value vectors produced for previous tokens so that they do not need to be recomputed during subsequent decode steps.

The KV cache memory required for one token across all transformer layers is:

\[
\text{KV bytes/token} = 2 \times L \times H_{kv} \times D \times S
\]

Where:

- \(2\): Key and Value tensors.
- \(L\): Number of transformer layers.
- \(H_{kv}\): Number of KV heads per layer.
- \(D\): Head dimension.
- \(S\): Bytes per element.

For Qwen2.5-0.5B using FP16 KV storage:

\[
2 \times 24 \times 2 \times 64 \times 2 = 12,288\text{ bytes}
\]

Therefore:

```text
KV bytes per token: 12,288
KV KiB per token: 12.00
```

Every additional token requires approximately **12 KiB of KV cache storage** across all layers.

This calculation accounts for the K/V tensors themselves, excluding allocator metadata and other inference memory overhead.

## 3. Introducing Fixed-Size KV Blocks

Instead of reserving one large contiguous KV memory region for every request, paged KV cache management divides storage into **fixed-size blocks**.

Each block can hold the KV cache for a fixed number of tokens.

For this notebook:

```text
BLOCK_SIZE = 16 tokens
```

The memory required for one block is:

\[
\text{KV bytes/block} = \text{KV bytes/token} \times \text{Block Size}
\]

For our model:

\[
12,288 \times 16 = 196,608\text{ bytes}
\]

Output:

```text
Tokens per block: 16
KV memory per block: 196,608 bytes
KV memory per block: 192.00 KiB
```

A physical block therefore represents **192 KiB of KV storage** across all transformer layers.

The important distinction is that blocks belonging to the same request do not need to occupy consecutive physical memory locations.

The inference engine maintains a block table that maps the request's logical blocks to physical KV cache blocks.

We'll implement that mapping later in this notebook.

## 4. Calculating Blocks Required per Request

Because KV cache allocation happens in fixed-size blocks, a request must receive enough blocks to accommodate its current token count.

The required block count is:

\[
\text{Blocks Required} = \left\lceil \frac{\text{Number of Tokens}}{\text{Block Size}} \right\rceil
\]

For example, a request containing 40 tokens requires:

\[
\left\lceil \frac{40}{16} \right\rceil = 3\text{ blocks}
\]

Those three blocks provide 48 token slots, leaving 8 unused slots in the final block.

Using the request lengths from our earlier scheduling experiments:

| Request | Tokens | Blocks allocated | Unused slots |
|---|---:|---:|---:|
| A | 40 | 3 | 8 |
| B | 43 | 3 | 5 |
| C | 38 | 3 | 10 |

Each request independently receives the blocks needed to store its KV cache.

### Why does this matter?

When generating tokens, a request's KV cache grows.

With block-based allocation, the request can continue using its existing blocks until the last block is full. Only then does it require another physical block.

This avoids having to reserve KV storage for the request's maximum possible sequence length upfront.

## 5. Understanding Internal Fragmentation

Fixed-size block allocation introduces a small amount of **internal fragmentation**.

Internal fragmentation occurs when a request occupies a physical block but does not use every token slot within that block.

For example:

```text
Request A: 40 tokens

Block 0: [16 / 16 slots used]
Block 1: [16 / 16 slots used]
Block 2: [ 8 / 16 slots used]

Total allocated capacity: 48 tokens
Actual tokens: 40
Unused capacity: 8 slots
```

The final block is only partially occupied, but the allocator must reserve the entire block.

For a nonempty request, the unused capacity in its last block is always between 0 and `BLOCK_SIZE - 1` token slots.

Smaller blocks can reduce this internal fragmentation, although block size also affects allocation metadata and attention-kernel design.

## 6. Measuring KV Block Utilization

We calculated the total KV storage required for our three requests.

```text
Actual tokens: 121
Allocated blocks: 9
Allocated token slots: 144
Unused token slots: 23
Allocated KV memory: 1728.00 KiB
Useful KV storage: 1452.00 KiB
Block utilization: 84.03%
```

We allocated 144 token slots to store 121 actual tokens.

The remaining 23 slots represent unused capacity inside the allocated blocks.

Block utilization is calculated as:

\[
\text{Block Utilization} =
\frac{\text{Actual Tokens}}{\text{Allocated Token Slots}} \times 100
\]

\[
= \frac{121}{144} \times 100 = 84.03\%
\]

This metric represents **how efficiently the allocated KV block capacity is being used**. It is not the GPU's compute utilization or overall VRAM utilization.

In a real inference engine, the block allocator also has to account for the availability of physical blocks across all active requests.

## What We've Learned So Far

We now understand how the architecture of a transformer determines its KV cache memory requirements and how those requirements translate into fixed-size physical blocks.

For our Qwen2.5-0.5B configuration:

- KV memory per token: **12 KiB**
- Tokens per block: **16**
- KV memory per block: **192 KiB**
- Blocks needed for requests A, B, and C: **9**
- Allocated KV block utilization: **84.03%**

We also established why fixed-size blocks are useful: requests can grow incrementally without reserving their maximum sequence length upfront, and their logical KV blocks can map to noncontiguous physical storage.

### Next: Physical Block Pool & Free-List Allocation

So far, we've only calculated how many blocks requests need.

Next, we'll implement the actual memory-management mechanism:

- A finite pool of physical KV blocks.
- A free list tracking available blocks.
- Block allocation as requests arrive and grow.
- Block release when requests finish.
- Per-request logical-to-physical block tables.

This will turn our memory calculations into a working **paged KV cache allocator**, which we can later integrate with our continuous batching scheduler.

Actual K/V tensor storage and attention computation will be covered in **Part 7B — PagedAttention**.

In [8]:
#writing the block pool


from collections import deque

class BlockPool:
    def __init__(self, num_blocks: int):
        if num_blocks <= 0:
            raise ValueError("Pool must contain at least one block")

        self.num_blocks = num_blocks

        # Physical IDs available for allocation
        self.free_blocks = deque(range(num_blocks))

        # Physical IDs currently allocated
        self.allocated_blocks = set()

    @property
    def num_free_blocks(self):
        return len(self.free_blocks)

    @property
    def num_allocated_blocks(self):
        return len(self.allocated_blocks)

    def allocate(self, num_blocks: int):
        if num_blocks < 0:
            raise ValueError("Cannot allocate negative blocks")

        if num_blocks > self.num_free_blocks:
            return None

        # Pop free physical blocks
        block_ids = [
            self.free_blocks.popleft()
            for _ in range(num_blocks)
        ]

        self.allocated_blocks.update(block_ids)
        return block_ids

    def free(self, block_ids):
        block_ids = list(block_ids)

        if len(block_ids) != len(set(block_ids)):
            raise ValueError("Duplicate block IDs")

        if not set(block_ids).issubset(self.allocated_blocks):
            raise ValueError("Attempted to free unallocated blocks")

        for block_id in block_ids:
            self.allocated_blocks.remove(block_id)
            self.free_blocks.appendleft(block_id)

    def status(self):
        print(f"Total blocks: {self.num_blocks}")
        print(f"Allocated blocks: {self.num_allocated_blocks}")
        print(f"Free blocks: {self.num_free_blocks}")
        print(f"Free block IDs: {list(self.free_blocks)}")
        print(f"Allocated block IDs: {sorted(self.allocated_blocks)}")


In [9]:

pool = BlockPool(num_blocks=8)

pool.status()

print(f"\nTotal KV capacity: {8 * BLOCK_SIZE} token slots")
print(f"Total KV memory: {8 * kv_bytes_per_block / 1024:.2f} KiB")


Total blocks: 8
Allocated blocks: 0
Free blocks: 8
Free block IDs: [0, 1, 2, 3, 4, 5, 6, 7]
Allocated block IDs: []

Total KV capacity: 128 token slots
Total KV memory: 1536.00 KiB


In [10]:
#allocating blocks to requests


a_blocks = pool.allocate(blocks_required(request_lengths["A"]))
b_blocks = pool.allocate(blocks_required(request_lengths["B"]))

print("Request A blocks:", a_blocks)
print("Request B blocks:", b_blocks)

print("\nPool status:")
pool.status()


Request A blocks: [0, 1, 2]
Request B blocks: [3, 4, 5]

Pool status:
Total blocks: 8
Allocated blocks: 6
Free blocks: 2
Free block IDs: [6, 7]
Allocated block IDs: [0, 1, 2, 3, 4, 5]


In [11]:
#attemption more allocation even if capacity is insufficient


c_blocks = pool.allocate(blocks_required(request_lengths["C"]))

print("Request C allocation:", c_blocks)
print("Free blocks remaining:", pool.num_free_blocks)


Request C allocation: None
Free blocks remaining: 2


In [12]:
#freeing a block


pool.free(a_blocks)

print("Released Request A blocks")
pool.status()


Released Request A blocks
Total blocks: 8
Allocated blocks: 3
Free blocks: 5
Free block IDs: [2, 1, 0, 6, 7]
Allocated block IDs: [3, 4, 5]


In [13]:
#retrying c's allocation


c_blocks = pool.allocate(blocks_required(request_lengths["C"]))

print("Request C blocks:", c_blocks)
pool.status()


Request C blocks: [2, 1, 0]
Total blocks: 8
Allocated blocks: 6
Free blocks: 2
Free block IDs: [6, 7]
Allocated block IDs: [0, 1, 2, 3, 4, 5]


In the previous sections, we calculated how much KV cache memory each token and physical block requires.

For Qwen2.5-0.5B using FP16 KV storage:

- Each token requires **12 KiB** of KV cache memory.
- Each physical block holds **16 tokens**.
- Each physical block requires **192 KiB** of memory.

We also established that requests do not need to reserve memory for their maximum possible sequence length. Instead, they can acquire additional KV blocks as their sequences grow.

Now we need a mechanism to manage those physical blocks.

### 1. Understanding the Physical Block Pool

In a real inference engine, multiple requests share a finite amount of GPU memory available for KV caching.

We cannot allocate an unlimited number of KV blocks. The engine needs to track which blocks are currently occupied and which are available for reuse.

A **physical block pool** manages this allocation.

For example, suppose we have a pool containing 8 physical KV blocks:

```text
Physical Block Pool

Block IDs:  0  1  2  3  4  5  6  7
Status:     F  F  F  F  F  F  F  F

F = Free
```

When Request A needs 3 blocks, the allocator can assign:

```text
Request A: [0, 1, 2]

Block IDs:  0  1  2  3  4  5  6  7
Status:     A  A  A  F  F  F  F  F
```

Those blocks are now unavailable to other independent requests.

When Request A finishes, its allocated blocks can be returned to the pool and reused.

The important distinction is that our allocator manages **block IDs and allocation state**, not actual GPU tensors. We'll connect these concepts to physical KV storage when implementing PagedAttention.

### 2. Implementing the BlockPool Class

We implemented a simple `BlockPool` class to handle block allocation and release.

The design follows the basic block-pool mechanism used in inference engines such as [vLLM](https://github.com/vllm-project/vllm), although production implementations also support more advanced features such as reference counting and prefix caching.

#### Initializing the Pool

```python
def __init__(self, num_blocks: int):
    if num_blocks <= 0:
        raise ValueError("Pool must contain at least one block")

    self.num_blocks = num_blocks

    self.free_blocks = deque(range(num_blocks))
    self.allocated_blocks = set()
```

We maintain two collections.

**`free_blocks`**

A `deque` containing the IDs of physical blocks available for allocation.

When we initialize a pool with 8 blocks:

```python
free_blocks = deque([0, 1, 2, 3, 4, 5, 6, 7])
```

We use `deque` because it supports efficient operations at both ends of the collection.

This is useful because our allocator removes available blocks from the front and returns released blocks to the front.

**`allocated_blocks`**

A set containing the IDs of physical blocks that are currently allocated.

Initially, this set is empty:

```python
allocated_blocks = set()
```

We use a set because it supports efficient membership checks and prevents duplicate entries.

Together, these collections describe the allocation state of the entire block pool.

#### Tracking Available Capacity

We also introduced two properties:

```python
@property
def num_free_blocks(self):
    return len(self.free_blocks)

@property
def num_allocated_blocks(self):
    return len(self.allocated_blocks)
```

The `@property` decorator allows us to access these values like attributes:

```python
pool.num_free_blocks
pool.num_allocated_blocks
```

Instead of calling them as methods.

These properties are useful for the scheduler because it needs to know how many blocks remain available before admitting or growing a request.

For our simple allocator, the following relationship should always hold:

\[
\text{Free Blocks} + \text{Allocated Blocks} = \text{Total Blocks}
\]

### 3. Allocating Physical Blocks

The allocation method takes the number of blocks requested:

```python
def allocate(self, num_blocks: int):
    if num_blocks < 0:
        raise ValueError("Cannot allocate negative blocks")

    if num_blocks > self.num_free_blocks:
        return None

    block_ids = [
        self.free_blocks.popleft()
        for _ in range(num_blocks)
    ]

    self.allocated_blocks.update(block_ids)
    return block_ids
```

Let's understand the execution.

**Step 1: Validate the allocation request**

The allocator rejects negative block counts.

**Step 2: Check available capacity**

```python
if num_blocks > self.num_free_blocks:
    return None
```

If there are not enough free blocks, the allocation fails without changing the pool.

For example, if only 2 blocks are available but a request needs 3, the method returns `None`.

This is an **all-or-nothing allocation policy**. The allocator does not partially allocate the requested blocks.

**Step 3: Remove blocks from the free list**

```python
self.free_blocks.popleft()
```

`popleft()` removes and returns the first available block ID.

For a pool containing:

```text
Free blocks: [0, 1, 2, 3, 4, 5, 6, 7]
```

Allocating 3 blocks produces:

```text
Allocated:   [0, 1, 2]
Remaining:   [3, 4, 5, 6, 7]
```

**Step 4: Update allocation state**

```python
self.allocated_blocks.update(block_ids)
```

The allocated IDs are added to the set of occupied blocks.

Finally, the allocator returns their IDs to the caller.

An important observation is that **the allocator does not require consecutive physical block IDs**. It simply assigns available blocks from the free list.

This will become important when we introduce per-request block tables.

### 4. Releasing Physical Blocks

When a request finishes, its KV cache is no longer required for further decoding.

In our non-caching implementation, the blocks can be returned to the free pool.

```python
def free(self, block_ids):
    block_ids = list(block_ids)

    if len(block_ids) != len(set(block_ids)):
        raise ValueError("Duplicate block IDs")

    if not set(block_ids).issubset(self.allocated_blocks):
        raise ValueError("Attempted to free unallocated blocks")

    for block_id in block_ids:
        self.allocated_blocks.remove(block_id)
        self.free_blocks.appendleft(block_id)
```

The method performs three operations.

**1. Validate the block IDs**

We reject duplicate IDs and ensure every block being released is currently allocated.

This prevents accidental double-free operations.

**2. Remove the blocks from the allocated set**

```python
self.allocated_blocks.remove(block_id)
```

The block is no longer considered occupied.

**3. Return the block to the free list**

```python
self.free_blocks.appendleft(block_id)
```

`appendleft()` inserts the released block at the front of the deque.

This means recently released blocks will be allocated again before older free blocks, following the free-list behavior chosen for this implementation.

For example:

```text
Before releasing A:

Allocated: [0, 1, 2, 3, 4, 5]
Free:      [6, 7]

Release A's blocks: [0, 1, 2]

After releasing A:

Allocated: [3, 4, 5]
Free:      [2, 1, 0, 6, 7]
```

Notice that the returned IDs appear in reverse order at the front.

This happens because each block is inserted using `appendleft()`.

The allocator does not clear GPU memory when a block is released. It only updates allocation metadata. Any previous KV contents must be treated as invalid for the new owner.

### 5. Creating a Finite Block Pool

We initialized our allocator with 8 physical blocks:

```python
pool = BlockPool(num_blocks=8)
```

Since each block can hold 16 tokens:

\[
8 \times 16 = 128 \text{ token slots}
\]

And each block requires 192 KiB:

\[
8 \times 192 = 1536 \text{ KiB}
\]

Our pool therefore represents **1.5 MiB of KV cache capacity**.

Output:

```text
Total blocks: 8
Allocated blocks: 0
Free blocks: 8
Free block IDs: [0, 1, 2, 3, 4, 5, 6, 7]
Allocated block IDs: []

Total KV capacity: 128 token slots
Total KV memory: 1536.00 KiB
```

This is a deliberately small pool so we can observe how requests compete for limited KV capacity.

No actual 1.5 MiB GPU allocation is performed at this stage.

### 6. Allocating Blocks for Multiple Requests

We used the same request lengths from our previous scheduling experiments.

| Request | Tokens | Required blocks |
|---|---:|---:|
| A | 40 | 3 |
| B | 43 | 3 |
| C | 38 | 3 |

We first allocated blocks for Requests A and B.

```python
a_blocks = pool.allocate(blocks_required(request_lengths["A"]))
b_blocks = pool.allocate(blocks_required(request_lengths["B"]))
```

The resulting allocations were:

```text
Request A blocks: [0, 1, 2]
Request B blocks: [3, 4, 5]

Total blocks: 8
Allocated blocks: 6
Free blocks: 2
Free block IDs: [6, 7]
Allocated block IDs: [0, 1, 2, 3, 4, 5]
```

At this point, Requests A and B collectively occupy 6 physical blocks.

Only 2 remain available.

### 7. Handling Insufficient KV Capacity

Next, we attempted to allocate blocks for Request C.

Request C contains 38 tokens and therefore needs:

\[
\left\lceil \frac{38}{16} \right\rceil = 3 \text{ blocks}
\]

However, the pool only has 2 free blocks.

```python
c_blocks = pool.allocate(blocks_required(request_lengths["C"]))
```

Output:

```text
Request C allocation: None
Free blocks remaining: 2
```

The allocation failed because the requested number of blocks exceeded the available capacity.

Importantly, the allocator did not consume the remaining 2 blocks.

This is relevant to inference scheduling.

When the memory manager cannot satisfy an allocation request, the scheduler must decide what to do next.

Depending on the scheduling policy, it could defer the request until memory becomes available or preempt an existing request to reclaim capacity.

Our allocator does not make those scheduling decisions. It only reports whether allocation succeeds.

### 8. Releasing and Reusing Blocks

Suppose Request A finishes generating its response.

We can release its KV blocks:

```python
pool.free(a_blocks)
```

Output:

```text
Released Request A blocks
Total blocks: 8
Allocated blocks: 3
Free blocks: 5
Free block IDs: [2, 1, 0, 6, 7]
Allocated block IDs: [3, 4, 5]
```

We now have 5 free physical blocks.

Request C can attempt allocation again:

```python
c_blocks = pool.allocate(blocks_required(request_lengths["C"]))
```

Output:

```text
Request C blocks: [2, 1, 0]
Total blocks: 8
Allocated blocks: 6
Free blocks: 2
Free block IDs: [6, 7]
Allocated block IDs: [0, 1, 2, 3, 4, 5]
```

Request C successfully acquired the physical blocks previously occupied by Request A.

An important detail is that the blocks were assigned in the order `[2, 1, 0]`.

This does not cause a problem.

A request's logical sequence order does not depend on its physical block IDs being consecutive or increasing.

For example:

```text
Request C

Logical Block 0 -> Physical Block 2
Logical Block 1 -> Physical Block 1
Logical Block 2 -> Physical Block 0
```

The request still has a logical sequence of 38 tokens, even though its physical storage blocks are assigned in a different order.

Later, the block table will preserve these mappings so that the attention computation can locate the correct K/V data.

## What We've Implemented

We've built a basic physical KV block allocator that can:

- Maintain a finite pool of physical blocks.
- Track allocated and available blocks.
- Allocate blocks based on a request's requirements.
- Reject allocations when insufficient capacity exists.
- Reclaim blocks when requests finish.
- Reuse released physical blocks for other requests.

The key takeaway is that **physical KV block allocation is separate from the logical token sequence of a request**.

Our current allocator tracks which blocks are available, but it does not yet maintain request-specific block tables or dynamically grow allocations as decoding progresses.

### Next: 7A.4 — Per-Request Block Tables and Dynamic KV Growth

Next, we'll build a request-level memory manager on top of our existing `BlockPool`.

Each request will maintain its own logical-to-physical block mapping.

We'll then implement incremental allocation as a request's sequence length increases, including the point where its final allocated block becomes full and an additional physical block is required.

This will give us the foundation needed to integrate paged KV allocation into our continuous batching scheduler.

In [14]:
#creating class to manage requests and what blocks will it manage


class RequestBlockManager:
    def __init__(self, pool: BlockPool, block_size: int):
        self.pool = pool
        self.block_size = block_size

        # request_id -> list of physical block IDs
        self.block_tables = {}

        # request_id -> number of KV token slots in use
        self.token_counts = {}

    def blocks_needed(self, num_tokens):
        return (
            num_tokens + self.block_size - 1
        ) // self.block_size

    def add_request(self, request_id, num_tokens):
        if request_id in self.block_tables:
            raise ValueError("Request already exists")

        if num_tokens < 0:
            raise ValueError("Negative token count")

        required = self.blocks_needed(num_tokens)
        physical_blocks = self.pool.allocate(required)

        if physical_blocks is None:
            return False

        self.block_tables[request_id] = physical_blocks
        self.token_counts[request_id] = num_tokens
        return True

    def grow_request(self, request_id, new_token_count):
        if request_id not in self.block_tables:
            raise ValueError("Unknown request")

        current_tokens = self.token_counts[request_id]

        if new_token_count < current_tokens:
            raise ValueError("Cannot shrink with grow_request")

        current_blocks = len(self.block_tables[request_id])
        required_blocks = self.blocks_needed(new_token_count)

        additional_blocks = required_blocks - current_blocks

        if additional_blocks > 0:
            new_blocks = self.pool.allocate(additional_blocks)

            if new_blocks is None:
                return False

            self.block_tables[request_id].extend(new_blocks)

        self.token_counts[request_id] = new_token_count
        return True

    def locate_token(self, request_id, token_index):
        if request_id not in self.block_tables:
            raise ValueError("Unknown request")

        if not 0 <= token_index < self.token_counts[request_id]:
            raise IndexError("Token outside recorded KV range")

        logical_block = token_index // self.block_size
        offset = token_index % self.block_size

        physical_block = self.block_tables[request_id][logical_block]

        return physical_block, offset

    def release_request(self, request_id):
        if request_id not in self.block_tables:
            raise ValueError("Unknown request")

        blocks = self.block_tables[request_id]
        self.pool.free(blocks)

        del self.block_tables[request_id]
        del self.token_counts[request_id]

    def status(self):
        for request_id, blocks in self.block_tables.items():
            print(
                f"{request_id}: "
                f"tokens={self.token_counts[request_id]}, "
                f"blocks={blocks}"
            )

        print("Free physical blocks:", self.pool.num_free_blocks)


In [15]:
#creating reqs

pool = BlockPool(num_blocks=8)

manager = RequestBlockManager(
    pool=pool,
    block_size=BLOCK_SIZE
)

manager.add_request("A", 40)
manager.add_request("B", 43)

manager.status()


A: tokens=40, blocks=[0, 1, 2]
B: tokens=43, blocks=[3, 4, 5]
Free physical blocks: 2


In [16]:

request_id = "A"
token_index = 35

physical_block, offset = manager.locate_token(
    request_id,
    token_index
)

print("Request:", request_id)
print("Token index:", token_index)
print("Logical block:", token_index // BLOCK_SIZE)
print("Physical block:", physical_block)
print("Offset inside block:", offset)


Request: A
Token index: 35
Logical block: 2
Physical block: 2
Offset inside block: 3


In [17]:
#Grow Request A to 48 tokens


success = manager.grow_request("A", 48)

print("Growth successful:", success)
manager.status()


Growth successful: True
A: tokens=48, blocks=[0, 1, 2]
B: tokens=43, blocks=[3, 4, 5]
Free physical blocks: 2


In [18]:
#Grow Request A to 49 tokens


success = manager.grow_request("A", 49)

print("Growth successful:", success)
manager.status()


Growth successful: True
A: tokens=49, blocks=[0, 1, 2, 6]
B: tokens=43, blocks=[3, 4, 5]
Free physical blocks: 1


In [19]:
#Growing Request B


success = manager.grow_request("B", 49)

print("Growth successful:", success)
manager.status()


Growth successful: True
A: tokens=49, blocks=[0, 1, 2, 6]
B: tokens=49, blocks=[3, 4, 5, 7]
Free physical blocks: 0


In [20]:
#Attempt to admit Request C


success = manager.add_request("C", 38)

print("Request C admitted:", success)
manager.status()


Request C admitted: False
A: tokens=49, blocks=[0, 1, 2, 6]
B: tokens=49, blocks=[3, 4, 5, 7]
Free physical blocks: 0


In [21]:
#Release Request A


manager.release_request("A")

print("Released Request A")
manager.status()


Released Request A
B: tokens=49, blocks=[3, 4, 5, 7]
Free physical blocks: 4


In [22]:
#now Admit Request C


success = manager.add_request("C", 38)

print("Request C admitted:", success)
manager.status()


Request C admitted: True
B: tokens=49, blocks=[3, 4, 5, 7]
C: tokens=38, blocks=[6, 2, 1]
Free physical blocks: 1




In the previous section, we implemented a `BlockPool` capable of allocating, freeing, and reusing physical KV cache blocks.

Our allocator could track which physical blocks were occupied and which were available. However, it did not maintain information about individual requests.

For example, if the allocator reported that physical blocks `[0, 1, 2, 3, 4, 5]` were allocated, we had no internal mapping telling us which request owned each block or which token positions those blocks represented.

To solve this, we need **per-request block tables**.

Block tables are a fundamental component of paged KV cache management. They allow inference engines to maintain logically sequential KV cache positions even when the corresponding data is stored in noncontiguous physical memory blocks.

In this section, we'll build a `RequestBlockManager` on top of our existing `BlockPool`.

Our implementation will support:

1. Maintaining a separate block table for each request.
2. Tracking the number of KV token positions associated with each request.
3. Translating logical token indices into physical block IDs and offsets.
4. Dynamically allocating additional KV blocks as requests grow.
5. Handling insufficient memory without modifying existing allocations.
6. Releasing and reusing physical blocks when requests finish.

The underlying concepts are based on block-based KV cache management used in inference engines such as [vLLM](https://github.com/vllm-project/vllm).

### 1. Understanding Logical and Physical KV Blocks

Before implementing the request manager, we need to distinguish between two concepts.

**Logical blocks**

Logical blocks represent consecutive groups of token positions belonging to a request.

Suppose Request A has 40 tokens and our block size is 16.

The request requires:

\[
\left\lceil \frac{40}{16} \right\rceil = 3 \text{ blocks}
\]

Its logical block structure looks like:

```text
Request A: 40 tokens

Logical Block 0: Tokens 0-15
Logical Block 1: Tokens 16-31
Logical Block 2: Tokens 32-39
```

The third block has capacity for 16 tokens but currently uses only 8 positions.

Logical block numbers describe the order of token positions in the sequence. They do not specify where the KV data is stored physically.

**Physical blocks**

Physical blocks represent fixed-size storage regions in the KV cache memory pool.

For example, the allocator might assign physical blocks `[7, 2, 5]` to Request A.

The mapping becomes:

```text
Request A

Logical Block 0 -> Physical Block 7
Logical Block 1 -> Physical Block 2
Logical Block 2 -> Physical Block 5
```

Although the physical IDs are not consecutive, the request still has a logically continuous sequence of 40 tokens.

This is possible because we maintain a mapping between logical and physical blocks.

### 2. Introducing the Block Table

A block table is an ordered collection of physical block IDs belonging to a request.

For example:

```python
block_table = [7, 2, 5]
```

The position of each element represents the logical block index.

The value at that position represents the corresponding physical block ID.

```text
Block table:

Logical index:      0    1    2
                    |    |    |
Physical block:     7    2    5
```

When the attention computation needs the KV data corresponding to a specific token, it can use this mapping to locate the correct physical block.

**Example: Locating token 35**

Our block size is 16.

First, calculate the logical block index:

\[
\text{Logical Block} = \left\lfloor \frac{\text{Token Index}}{\text{Block Size}} \right\rfloor
\]

For token 35:

\[
35 // 16 = 2
\]

Next, calculate the token's offset within that logical block:

\[
\text{Offset} = \text{Token Index} \bmod \text{Block Size}
\]

\[
35 \bmod 16 = 3
\]

Finally, look up the physical block:

```python
physical_block = block_table[2]
```

Since our example block table is `[7, 2, 5]`, the result is:

```text
Token index: 35
Logical block: 2
Physical block: 5
Offset inside block: 3
```

This gives us the information needed to locate the token's KV data in a paged storage layout.

Importantly, we do not need to rearrange the actual token sequence or require physical blocks to be consecutive.

The block table handles the translation.

In Part 7B, we'll use this same addressing principle when implementing attention over paged K/V storage.

### 3. Implementing the RequestBlockManager

Our previous `BlockPool` class manages physical block availability.

Now we introduce a second component called `RequestBlockManager`.

The responsibilities are separate:

| Component | Responsibility |
|---|---|
| `BlockPool` | Tracks and allocates available physical block IDs |
| `RequestBlockManager` | Tracks request-specific block mappings and KV token counts |

The request manager will use the existing allocator whenever a request needs additional physical blocks.

#### Initializing the RequestBlockManager

We begin with:

```python
class RequestBlockManager:
    def __init__(self, pool: BlockPool, block_size: int):
        self.pool = pool
        self.block_size = block_size

        self.block_tables = {}
        self.token_counts = {}
```

Let's understand each attribute.

**`self.pool`**

This holds a reference to the existing `BlockPool` instance.

```python
self.pool = pool
```

The request manager does not create its own independent physical memory pool.

Instead, every request shares the same allocator.

This ensures that physical block availability is coordinated across all requests.

**`self.block_size`**

```python
self.block_size = block_size
```

This represents the number of token positions each physical KV block can accommodate.

For our experiment:

```python
BLOCK_SIZE = 16
```

**`self.block_tables`**

```python
self.block_tables = {}
```

This dictionary maps each request ID to an ordered list of physical blocks.

For example:

```python
{
    "A": [0, 1, 2],
    "B": [3, 4, 5]
}
```

Request A owns physical blocks `[0, 1, 2]`.

Request B owns physical blocks `[3, 4, 5]`.

The order of these lists represents the logical block ordering of each request.

**`self.token_counts`**

```python
self.token_counts = {}
```

This dictionary tracks the number of KV token positions represented by each request.

For example:

```python
{
    "A": 40,
    "B": 43
}
```

Why do we need this when we already have block tables?

Because a block table tells us how much capacity has been allocated, not how many positions are currently occupied.

Request A has 3 blocks:

\[
3 \times 16 = 48 \text{ token slots}
\]

But it currently represents only 40 KV token positions.

The token count tells us that the final block has 8 occupied positions and 8 unused positions.

This information becomes especially important when sequences grow during decoding.

In our current notebook, token counts are simulated committed KV positions. We are not yet computing or storing actual Key and Value tensors.

### 4. Calculating Required Blocks

Inside `RequestBlockManager`, we implemented:

```python
def blocks_needed(self, num_tokens):
    return (
        num_tokens + self.block_size - 1
    ) // self.block_size
```

This is the integer arithmetic equivalent of:

\[
\left\lceil\frac{\text{num\_tokens}}{\text{block\_size}}\right\rceil
\]

We use ceiling division because even a partially occupied block requires an entire physical block allocation.

For example:

| Tokens | Blocks required |
|---|---:|
| 1 | 1 |
| 16 | 1 |
| 17 | 2 |
| 32 | 2 |
| 40 | 3 |
| 48 | 3 |
| 49 | 4 |

Notice the boundary between 48 and 49 tokens.

A request with 48 tokens fits exactly inside three blocks.

Once the request grows to 49 tokens, it requires a fourth block.

This boundary is central to dynamic KV cache allocation.

### 5. Admitting a New Request

The `add_request()` method creates a block table for a new request.

```python
def add_request(self, request_id, num_tokens):
    if request_id in self.block_tables:
        raise ValueError("Request already exists")

    if num_tokens < 0:
        raise ValueError("Negative token count")

    required = self.blocks_needed(num_tokens)
    physical_blocks = self.pool.allocate(required)

    if physical_blocks is None:
        return False

    self.block_tables[request_id] = physical_blocks
    self.token_counts[request_id] = num_tokens
    return True
```

Let's follow the execution.

**Step 1: Check whether the request already exists**

```python
if request_id in self.block_tables:
    raise ValueError("Request already exists")
```

Every active request must have a unique identifier.

This prevents accidentally overwriting an existing request's block table and losing track of its physical allocations.

**Step 2: Validate the token count**

Negative token counts are invalid.

**Step 3: Calculate the number of blocks required**

```python
required = self.blocks_needed(num_tokens)
```

For a request with 40 tokens and block size 16:

```text
Required blocks: 3
```

**Step 4: Request physical blocks**

```python
physical_blocks = self.pool.allocate(required)
```

The request manager delegates physical allocation to `BlockPool`.

If enough blocks are available, the allocator returns their IDs.

Otherwise, it returns `None`.

**Step 5: Record the request's allocation**

```python
self.block_tables[request_id] = physical_blocks
self.token_counts[request_id] = num_tokens
```

The request manager now knows which physical blocks belong to the request and how many KV token positions they represent.

If the allocation fails, the method returns `False` without inserting the request into either dictionary.

This prevents partially admitted requests from modifying the memory-management state.

### 6. Initializing Our Requests

We created a fresh physical block pool containing 8 blocks:

```python
pool = BlockPool(num_blocks=8)

manager = RequestBlockManager(
    pool=pool,
    block_size=BLOCK_SIZE
)
```

Each block stores 16 token positions.

Therefore, the total pool capacity is:

\[
8 \times 16 = 128 \text{ token slots}
\]

For Qwen2.5-0.5B using FP16 KV storage, this represents 1.5 MiB of KV block capacity.

We then admitted two requests:

```python
manager.add_request("A", 40)
manager.add_request("B", 43)
```

Output:

```text
A: tokens=40, blocks=[0, 1, 2]
B: tokens=43, blocks=[3, 4, 5]
Free physical blocks: 2
```

Let's understand the result.

Request A requires 3 blocks for 40 tokens.

Request B also requires 3 blocks for 43 tokens.

Together, they occupy 6 physical blocks.

That leaves 2 blocks available.

Their block tables are now managed internally by `RequestBlockManager`, rather than being stored in separate variables such as `a_blocks` and `b_blocks`.

### 7. Implementing Logical-to-Physical Token Addressing

The next important method is `locate_token()`.

```python
def locate_token(self, request_id, token_index):
    if request_id not in self.block_tables:
        raise ValueError("Unknown request")

    if not 0 <= token_index < self.token_counts[request_id]:
        raise IndexError("Token outside recorded KV range")

    logical_block = token_index // self.block_size
    offset = token_index % self.block_size

    physical_block = self.block_tables[request_id][logical_block]

    return physical_block, offset
```

This method translates a token's logical index into two values:

1. The physical KV block containing that token.
2. The token's offset within that block.

**Step 1: Validate the request**

We first verify that the request exists in our manager.

**Step 2: Validate the token index**

We ensure that the token index is within the currently recorded KV range.

For a request containing 40 tokens, valid zero-based indices are 0 through 39.

**Step 3: Calculate the logical block**

```python
logical_block = token_index // self.block_size
```

Integer division tells us which logical block contains the token.

**Step 4: Calculate the offset**

```python
offset = token_index % self.block_size
```

The modulo operation tells us where the token belongs within the block.

**Step 5: Look up the physical block**

```python
physical_block = self.block_tables[request_id][logical_block]
```

The block table translates the logical block index into its corresponding physical block ID.

#### Our Executed Example

We tested token index 35 for Request A:

```python
physical_block, offset = manager.locate_token(
    "A",
    35
)
```

Output:

```text
Request: A
Token index: 35
Logical block: 2
Physical block: 2
Offset inside block: 3
```

The physical block is 2 because Request A currently owns `[0, 1, 2]`.

If its block table were `[7, 2, 5]`, the same token would map to physical block 5.

The logical token index and offset do not change.

Only the physical block mapping changes.

This separation is what allows paged KV cache management to support noncontiguous physical storage.

### 8. Dynamic KV Cache Growth

So far, we've allocated blocks when requests first arrive.

However, inference requests do not remain the same length throughout generation.

During autoregressive decoding, the model processes additional tokens, and the amount of cached K/V data grows.

Eventually, the request may need more physical blocks.

Instead of reserving storage for the maximum possible sequence length, the allocator can assign additional blocks only when required.

We implemented this behavior in `grow_request()`.

```python
def grow_request(self, request_id, new_token_count):
    if request_id not in self.block_tables:
        raise ValueError("Unknown request")

    current_tokens = self.token_counts[request_id]

    if new_token_count < current_tokens:
        raise ValueError("Cannot shrink with grow_request")

    current_blocks = len(self.block_tables[request_id])
    required_blocks = self.blocks_needed(new_token_count)

    additional_blocks = required_blocks - current_blocks

    if additional_blocks > 0:
        new_blocks = self.pool.allocate(additional_blocks)

        if new_blocks is None:
            return False

        self.block_tables[request_id].extend(new_blocks)

    self.token_counts[request_id] = new_token_count
    return True
```

Let's understand the important logic.

**Step 1: Retrieve the current token count**

```python
current_tokens = self.token_counts[request_id]
```

This tells us how many KV positions the request currently represents.

**Step 2: Prevent shrinking through the growth method**

```python
if new_token_count < current_tokens:
    raise ValueError("Cannot shrink with grow_request")
```

Our method is specifically designed to increase or maintain a request's token count.

Shrinking or truncating cached sequences would require separate handling.

**Step 3: Calculate the required capacity**

```python
current_blocks = len(self.block_tables[request_id])
required_blocks = self.blocks_needed(new_token_count)
```

We compare the number of blocks already allocated with the number required for the updated token count.

**Step 4: Calculate additional block requirements**

```python
additional_blocks = required_blocks - current_blocks
```

For example:

```text
Current tokens: 40
Current blocks: 3

New tokens: 48
Required blocks: 3

Additional blocks: 0
```

The request can grow without allocating another block.

But if the request reaches 49 tokens:

```text
Current blocks: 3

New tokens: 49
Required blocks: 4

Additional blocks: 1
```

Now an additional physical block is required.

**Step 5: Allocate only the additional blocks**

```python
new_blocks = self.pool.allocate(additional_blocks)
```

The allocator only needs to satisfy the incremental growth requirement.

Previously allocated blocks remain unchanged.

**Step 6: Extend the block table**

```python
self.block_tables[request_id].extend(new_blocks)
```

The new physical block IDs are appended to the request's existing logical block mapping.

For example:

```text
Before growth:

Request A: [0, 1, 2]

After growth:

Request A: [0, 1, 2, 6]
```

The request now has one additional logical block mapped to physical block 6.

There is no need to relocate the previously allocated blocks.

**Step 7: Update the token count**

```python
self.token_counts[request_id] = new_token_count
```

This happens only after the required allocation succeeds.

If allocation fails, the method returns `False` without changing the token count or existing block table.

Our implementation models committed KV positions directly. In a real inference engine, allocation would be coordinated with successful prefill or decode computation.

### 9. Growing Request A From 40 to 48 Tokens

We first grew Request A from 40 tokens to 48 tokens.

```python
success = manager.grow_request("A", 48)
```

Output:

```text
Growth successful: True
A: tokens=48, blocks=[0, 1, 2]
B: tokens=43, blocks=[3, 4, 5]
Free physical blocks: 2
```

Notice that Request A still owns the same three physical blocks.

This is because:

\[
48 / 16 = 3
\]

All 48 KV positions fit inside its existing allocation.

No additional physical block was needed.

The free-block count remains unchanged.

This is an important optimization.

A request does not need to perform a new block allocation for every generated token.

It only needs another block when its existing allocated capacity becomes insufficient.

### 10. Growing Request A From 48 to 49 Tokens

Next, we increased Request A's token count to 49.

```python
success = manager.grow_request("A", 49)
```

Output:

```text
Growth successful: True
A: tokens=49, blocks=[0, 1, 2, 6]
B: tokens=43, blocks=[3, 4, 5]
Free physical blocks: 1
```

Request A now requires:

\[
\left\lceil\frac{49}{16}\right\rceil = 4 \text{ blocks}
\]

The allocator assigned physical block 6.

Its block table grew from:

```text
[0, 1, 2]
```

to:

```text
[0, 1, 2, 6]
```

The original three physical blocks remain in place.

Only the new block is appended.

This demonstrates one of the main advantages of paged KV cache management.

**Sequences can grow incrementally without requiring their existing KV blocks to be moved into a larger contiguous region.**

### 11. Growing Multiple Requests Independently

Next, we grew Request B from 43 tokens to 49 tokens.

```python
success = manager.grow_request("B", 49)
```

Output:

```text
Growth successful: True
A: tokens=49, blocks=[0, 1, 2, 6]
B: tokens=49, blocks=[3, 4, 5, 7]
Free physical blocks: 0
```

Request B also crossed the 48-token boundary.

It therefore required a fourth physical block.

The allocator assigned physical block 7.

Our pool is now fully allocated:

```text
Physical Block Pool

Block ID:   0  1  2  3  4  5  6  7
Owner:      A  A  A  B  B  B  A  B
```

Request A owns:

```text
[0, 1, 2, 6]
```

Request B owns:

```text
[3, 4, 5, 7]
```

Notice that both requests have noncontiguous physical block allocations.

Request A's fourth block is physically separated from its first three blocks.

Request B's fourth block is also separated from its initial allocation.

Their logical sequences remain intact because each request maintains its own ordered block table.

### 12. Handling Insufficient Memory During Request Admission

Once both requests reached 49 tokens, all 8 physical blocks were allocated.

We then attempted to admit Request C with 38 tokens.

```python
success = manager.add_request("C", 38)
```

Request C requires:

\[
\left\lceil\frac{38}{16}\right\rceil = 3 \text{ blocks}
\]

However, no physical blocks remain available.

Output:

```text
Request C admitted: False
A: tokens=49, blocks=[0, 1, 2, 6]
B: tokens=49, blocks=[3, 4, 5, 7]
Free physical blocks: 0
```

The request manager returned `False`.

Request C was not inserted into the block table dictionary.

The existing allocations for A and B remained unchanged.

This demonstrates an important principle of memory management.

**A failed allocation must not partially modify the request's memory state.**

Without this guarantee, the scheduler could lose track of physical blocks or incorrectly assume that a request has more KV capacity than it actually owns.

In a full inference engine, the scheduler would use this allocation failure to decide whether Request C should remain waiting or whether memory should be reclaimed through another mechanism.

We will integrate this behavior into our scheduler in the next section.

### 13. Releasing a Finished Request

Suppose Request A finishes generating its response.

Its KV cache is no longer needed for the active request.

We implemented the following method to release its allocated blocks:

```python
def release_request(self, request_id):
    if request_id not in self.block_tables:
        raise ValueError("Unknown request")

    blocks = self.block_tables[request_id]
    self.pool.free(blocks)

    del self.block_tables[request_id]
    del self.token_counts[request_id]
```

Let's examine what happens.

**Step 1: Retrieve the request's physical blocks**

```python
blocks = self.block_tables[request_id]
```

For Request A:

```text
[0, 1, 2, 6]
```

**Step 2: Return the blocks to the physical pool**

```python
self.pool.free(blocks)
```

The physical block allocator marks these IDs as available.

**Step 3: Delete the request's block table**

```python
del self.block_tables[request_id]
```

The request no longer owns any physical blocks.

**Step 4: Delete its token count**

```python
del self.token_counts[request_id]
```

We remove the request's KV length metadata.

This completes the release process.

#### Our Executed Example

We released Request A:

```python
manager.release_request("A")
```

Output:

```text
Released Request A
B: tokens=49, blocks=[3, 4, 5, 7]
Free physical blocks: 4
```

Four physical blocks became available.

Request B's allocation remained untouched.

This is important because requests can complete independently.

The completion of one request must not interfere with the KV cache belonging to other active requests.

### 14. Reusing Released Blocks for a New Request

After releasing Request A, we retried the admission of Request C.

```python
success = manager.add_request("C", 38)
```

Output:

```text
Request C admitted: True
B: tokens=49, blocks=[3, 4, 5, 7]
C: tokens=38, blocks=[6, 2, 1]
Free physical blocks: 1
```

Request C successfully acquired three physical blocks.

Its block table is:

```text
[6, 2, 1]
```

This happened because our `BlockPool` returns freed blocks to the front of its `deque` using `appendleft()`.

When Request A released `[0, 1, 2, 6]`, those blocks became available in reverse insertion order.

The allocator then assigned the first three available IDs to Request C.

The resulting mapping is:

```text
Request C: 38 tokens

Logical Block 0 -> Physical Block 6
Logical Block 1 -> Physical Block 2
Logical Block 2 -> Physical Block 1
```

Even though the physical block IDs are out of order, the request's KV token positions remain logically sequential.

For example:

```text
Logical Block 0:
Tokens 0-15
Physical Block 6

Logical Block 1:
Tokens 16-31
Physical Block 2

Logical Block 2:
Tokens 32-37
Physical Block 1
```

We can still locate any token using the same logical-to-physical translation:

\[
\text{Logical Block} = \text{Token Index} // 16
\]

\[
\text{Offset} = \text{Token Index} \bmod 16
\]

The physical block is retrieved from the request's block table.

This demonstrates why physical contiguity is not required for a logically continuous KV cache.

### 15. How the Two Components Work Together

We now have two separate memory-management components.

**BlockPool**

Responsible for physical memory allocation.

It maintains:

```python
free_blocks
allocated_blocks
```

It supports:

```python
allocate()
free()
```

**RequestBlockManager**

Responsible for request-specific KV mappings.

It maintains:

```python
block_tables
token_counts
```

It supports:

```python
add_request()
grow_request()
locate_token()
release_request()
```

The interaction looks like:

```text
                  Request A
                      |
                      v
             RequestBlockManager
                      |
              Determine blocks
                   required
                      |
                      v
                  BlockPool
                      |
             Allocate free IDs
                      |
                      v
             Physical Block IDs
                [0, 1, 2]
                      |
                      v
             RequestBlockManager
                      |
           Update A's block table
                      |
                      v
               A: [0, 1, 2]
```

When a request grows, the same process is repeated only if additional physical blocks are required.

When a request finishes, its blocks are returned to `BlockPool`, and its request-specific metadata is deleted.

This separation is useful because the block allocator does not need to understand individual token sequences.

It simply manages available storage units.

The request manager handles the relationship between those storage units and the logical KV positions of individual requests.

### 16. What This Implementation Does Not Handle Yet

Our implementation reproduces the basic allocation and mapping concepts of paged KV cache management, but it is still a simplified metadata-level simulation.

There are a few important differences from a production inference engine.

**Actual GPU KV storage**

We currently manage integer block IDs.

We have not allocated physical tensors to hold Key and Value vectors.

Those will be introduced in Part 7B.

**Coordination with model execution**

Our `grow_request()` method directly updates the recorded KV token count after allocation succeeds.

In a real inference engine, the scheduler must coordinate memory reservation with successful model execution.

A block can be reserved before computation, but its associated KV data is not valid until the model has actually written the corresponding K/V vectors.

**Shared KV blocks and reference counting**

Our implementation assumes that each physical block belongs to one request at a time.

Production engines can support sharing cached blocks between requests, particularly through prefix caching.

That requires additional mechanisms such as reference counting and cache-aware block reuse.

**Scheduling and preemption**

The request manager can report allocation failure, but it does not decide which request should execute next.

It also does not implement preemption.

Those decisions belong to the inference scheduler.

We'll address the relevant scheduling behavior when integrating our memory manager with the continuous batching concepts developed in the previous notebooks.

## What We've Learned

We started with a basic physical block allocator and extended it into a request-aware KV memory manager.

Our implementation now supports:

- Maintaining separate logical-to-physical block tables for concurrent requests.
- Calculating physical KV storage requirements from token counts.
- Admitting requests when sufficient physical blocks are available.
- Translating logical token indices into physical blocks and offsets.
- Growing KV allocations incrementally as requests process more tokens.
- Reusing partially filled final blocks before requesting additional storage.
- Preserving existing allocations when memory requests cannot be satisfied.
- Reclaiming physical blocks when requests finish.
- Reusing released blocks for other requests without requiring contiguous physical allocation.

The most important concept is that **a request's logical KV sequence is independent of the physical locations of its allocated KV blocks**.

This allows inference engines to manage KV memory in fixed-size units, allocate storage incrementally, and recycle physical blocks as requests arrive and finish.

### Next: 7A.5, Block-Aware Scheduler Integration

So far, we've manually called methods such as `add_request()`, `grow_request()`, and `release_request()`.

In a real inference engine, these operations are coordinated by the scheduler.

In the next section, we'll integrate our block manager with the continuous batching scheduler concepts from Part 6.

Instead of checking only a token-level KV capacity counter, the scheduler will account for physical block availability when deciding whether requests can execute.

We'll focus on coordinating block allocation with prefill, decode, request completion, and memory-blocked execution.

This will connect our memory-management implementation to the continuous batching workflow we built earlier.

In [23]:

from dataclasses import dataclass

# ============================================================
# 1. REQUEST STATE
# ============================================================

@dataclass
class SimRequest:
    request_id: str
    arrival_step: int
    prompt_tokens: int
    max_new_tokens: int

    prefilled: bool = False # i dint teach u this yet
    generated: int = 0
    state: str = "NOT_ARRIVED"


# ============================================================
# 2. BLOCK-AWARE SCHEDULER
# ============================================================

class BlockAwareScheduler:

    def __init__(self, pool, manager, token_budget, requests):
        self.pool = pool
        self.manager = manager
        self.token_budget = token_budget
        self.requests = requests

        self.step_num = 0
        self.history = []

        assert token_budget > 0
        assert len(set(r.request_id for r in requests)) == len(requests)
        assert all(
            r.prompt_tokens > 0
            and r.max_new_tokens > 0
            and r.arrival_step >= 1
            for r in requests
        )

    # --------------------------------------------------------
    # FINISH AND RELEASE KV BLOCKS
    # --------------------------------------------------------

    def finish_request(self, request, events):
        rid = request.request_id

        freed = self.manager.block_tables[rid].copy()
        self.manager.release_request(rid)

        request.state = "FINISHED"
        events.append(f"{rid}: FINISHED, freed blocks {freed}")

    # --------------------------------------------------------
    # VALIDATE MEMORY ACCOUNTING
    # --------------------------------------------------------

    def validate(self):
        owned = [
            b
            for table in self.manager.block_tables.values()
            for b in table
        ]

        assert len(owned) == len(set(owned))
        assert set(owned) == self.pool.allocated_blocks

        assert (
            self.pool.num_free_blocks
            + self.pool.num_allocated_blocks
            == self.pool.num_blocks
        )

        assert set(self.pool.free_blocks).isdisjoint(
            self.pool.allocated_blocks
        )

        assert len(self.pool.free_blocks) == len(
            set(self.pool.free_blocks)
        )

        for r in self.requests:
            rid = r.request_id

            if r.state == "RUNNING":
                assert rid in self.manager.block_tables

                # Last sampled output token has not yet
                # been processed into KV.
                expected_kv = (
                    r.prompt_tokens + r.generated - 1
                )

                assert self.manager.token_counts[rid] == expected_kv

                expected_blocks = self.manager.blocks_needed(
                    expected_kv
                )

                assert (
                    len(self.manager.block_tables[rid])
                    == expected_blocks
                )

            else:
                assert rid not in self.manager.block_tables

    # --------------------------------------------------------
    # EXECUTE ONE SCHEDULER ITERATION
    # --------------------------------------------------------

    def step(self):

        self.step_num += 1
        budget = self.token_budget
        events = []
        scheduled = []

        print(f"\n{'=' * 65}")
        print(f"SCHEDULER STEP {self.step_num}")
        print(f"{'=' * 65}")

        # ----------------------------------------------------
        # A. NEW REQUEST ARRIVALS
        # ----------------------------------------------------

        for r in self.requests:
            if (
                r.state == "NOT_ARRIVED"
                and r.arrival_step <= self.step_num
            ):
                r.state = "WAITING"
                events.append(
                    f"{r.request_id}: ARRIVED "
                    f"(prompt={r.prompt_tokens}, "
                    f"max_new={r.max_new_tokens})"
                )

        # ----------------------------------------------------
        # B. DECODE EXISTING RUNNING REQUESTS FIRST
        # ----------------------------------------------------

        for r in self.requests:

            if r.state != "RUNNING":
                continue

            if budget < 1:
                events.append(
                    f"{r.request_id}: decode deferred, "
                    "compute budget exhausted"
                )
                continue

            rid = r.request_id

            # A decode forward pass processes the most
            # recently sampled token and produces the next.
            #
            # Therefore KV grows by one computed token.

            current_kv = self.manager.token_counts[rid]
            target_kv = current_kv + 1

            success = self.manager.grow_request(
                rid, target_kv
            )

            if not success:
                events.append(
                    f"{rid}: KV-BLOCKED during decode "
                    f"(free_blocks={self.pool.num_free_blocks})"
                )
                continue

            # Simulated successful forward pass.
            budget -= 1
            r.generated += 1
            scheduled.append((rid, "DECODE", 1))

            events.append(
                f"{rid}: DECODE, "
                f"generated={r.generated}/{r.max_new_tokens}, "
                f"KV_tokens={target_kv}"
            )

            if r.generated >= r.max_new_tokens:
                self.finish_request(r, events)

        # ----------------------------------------------------
        # C. PREFILL WAITING REQUESTS
        # ----------------------------------------------------

        for r in self.requests:

            if r.state != "WAITING":
                continue

            rid = r.request_id

            # All-or-nothing prefill from Part 5B.
            if r.prompt_tokens > budget:
                events.append(
                    f"{rid}: PREFILL DEFERRED "
                    f"(needs={r.prompt_tokens}, budget={budget})"
                )

                # Preserve FIFO admission ordering.
                break

            # Allocate enough blocks for the entire prompt.
            success = self.manager.add_request(
                rid, r.prompt_tokens
            )

            if not success:
                events.append(
                    f"{rid}: KV-BLOCKED during prefill "
                    f"(needs_blocks="
                    f"{self.manager.blocks_needed(r.prompt_tokens)}, "
                    f"free_blocks={self.pool.num_free_blocks})"
                )

                # Preserve FIFO admission ordering.
                break

            # Simulated prefill forward pass.
            budget -= r.prompt_tokens
            r.prefilled = True
            r.state = "RUNNING"

            # Prefill computes prompt KV and samples the
            # first output token. That sampled token does
            # not have its own KV entry yet.
            r.generated = 1

            scheduled.append(
                (rid, "PREFILL", r.prompt_tokens)
            )

            events.append(
                f"{rid}: PREFILL completed "
                f"({r.prompt_tokens} tokens), "
                f"sampled first output token"
            )

            if r.generated >= r.max_new_tokens:
                self.finish_request(r, events)

        # ----------------------------------------------------
        # D. VALIDATE AND RECORD STEP
        # ----------------------------------------------------

        self.validate()

        snapshot = {
            "step": self.step_num,
            "scheduled": scheduled.copy(),
            "events": events.copy(),
            "budget_used": self.token_budget - budget,
            "free_blocks": self.pool.num_free_blocks,
            "allocated_blocks": self.pool.num_allocated_blocks,
            "block_tables": {
                rid: table.copy()
                for rid, table in self.manager.block_tables.items()
            }
        }

        self.history.append(snapshot)

        # ----------------------------------------------------
        # E. PRINT STEP SUMMARY
        # ----------------------------------------------------

        for event in events:
            print(" ", event)

        print("\nScheduled operations:")
        for rid, operation, tokens in scheduled:
            print(f"  {rid}: {operation} ({tokens} computed tokens)")

        print(
            f"\nCompute budget: "
            f"{self.token_budget - budget}/{self.token_budget}"
        )

        print(
            f"Physical KV blocks: "
            f"{self.pool.num_allocated_blocks}/{self.pool.num_blocks}"
        )

        print("Block tables:")
        if self.manager.block_tables:
            self.manager.status()
        else:
            print("  No active KV allocations")

        print("Request states:")
        for r in self.requests:
            print(
                f"  {r.request_id}: {r.state}, "
                f"generated={r.generated}/{r.max_new_tokens}"
            )

    # --------------------------------------------------------
    # F. RUN UNTIL ALL REQUESTS COMPLETE
    # --------------------------------------------------------

    def run(self, max_steps=100):

        for _ in range(max_steps):

            if all(
                r.state == "FINISHED"
                for r in self.requests
            ):
                break

            self.step()

        if not all(
            r.state == "FINISHED"
            for r in self.requests
        ):
            raise RuntimeError(
                "Simulation did not complete. "
                "Check compute budget, KV capacity, "
                "or blocked requests."
            )

        self.validate()

        assert self.pool.num_allocated_blocks == 0
        assert len(self.manager.block_tables) == 0

        print(f"\n{'=' * 65}")
        print("SIMULATION COMPLETE")
        print(f"{'=' * 65}")

        print("Total scheduling steps:", self.step_num)
        print("Free physical KV blocks:", self.pool.num_free_blocks)
        print("Allocated physical KV blocks:", self.pool.num_allocated_blocks)

        for r in self.requests:
            print(
                f"Request {r.request_id}: "
                f"{r.generated} output tokens, {r.state}"
            )


# ============================================================
# 3. INITIALIZE PHYSICAL KV MEMORY
# ============================================================

BLOCK_SIZE = 16

pool = BlockPool(num_blocks=8)

manager = RequestBlockManager(
    pool=pool,
    block_size=BLOCK_SIZE
)

# ============================================================
# 4. REQUEST WORKLOAD
# ============================================================

requests = [
    SimRequest(
        request_id="A",
        arrival_step=1,
        prompt_tokens=40,
        max_new_tokens=10
    ),
    SimRequest(
        request_id="B",
        arrival_step=2,
        prompt_tokens=43,
        max_new_tokens=7
    ),
    SimRequest(
        request_id="C",
        arrival_step=3,
        prompt_tokens=38,
        max_new_tokens=4
    )
]

# ============================================================
# 5. RUN THE SCHEDULER
# ============================================================

scheduler = BlockAwareScheduler(
    pool=pool,
    manager=manager,
    token_budget=50,
    requests=requests
)

scheduler.run(max_steps=100)



SCHEDULER STEP 1
  A: ARRIVED (prompt=40, max_new=10)
  A: PREFILL completed (40 tokens), sampled first output token

Scheduled operations:
  A: PREFILL (40 computed tokens)

Compute budget: 40/50
Physical KV blocks: 3/8
Block tables:
A: tokens=40, blocks=[0, 1, 2]
Free physical blocks: 5
Request states:
  A: RUNNING, generated=1/10
  B: NOT_ARRIVED, generated=0/7
  C: NOT_ARRIVED, generated=0/4

SCHEDULER STEP 2
  B: ARRIVED (prompt=43, max_new=7)
  A: DECODE, generated=2/10, KV_tokens=41
  B: PREFILL completed (43 tokens), sampled first output token

Scheduled operations:
  A: DECODE (1 computed tokens)
  B: PREFILL (43 computed tokens)

Compute budget: 44/50
Physical KV blocks: 6/8
Block tables:
A: tokens=41, blocks=[0, 1, 2]
B: tokens=43, blocks=[3, 4, 5]
Free physical blocks: 2
Request states:
  A: RUNNING, generated=2/10
  B: RUNNING, generated=1/7
  C: NOT_ARRIVED, generated=0/4

SCHEDULER STEP 3
  C: ARRIVED (prompt=38, max_new=4)
  A: DECODE, generated=3/10, KV_tokens=42
  B:

# 7A.5: Block-Aware Continuous Batching Scheduler

In the previous sections, we built two components for managing paged KV cache memory:

1. **`BlockPool`**, which manages the allocation and release of physical KV blocks.
2. **`RequestBlockManager`**, which maintains per-request block tables, tracks KV token counts, and handles dynamic block allocation.

However, until now, we manually called methods such as `add_request()`, `grow_request()`, and `release_request()`.

In a real inference engine, these operations are coordinated by the **scheduler**.

The scheduler must determine which requests can execute during each iteration while considering available compute capacity and KV cache memory.

In this section, we integrate our existing memory-management components into a block-aware continuous batching scheduler.

We will use the concepts already introduced in earlier notebooks:

- Continuous batching from Part 5A.
- Token-budget-aware scheduling from Part 5B.
- KV-capacity-aware scheduling from Part 6.
- Physical KV block allocation from Part 7A.3.
- Per-request block tables and dynamic growth from Part 7A.4.

We'll preserve our previous **all-or-nothing prefill policy**, meaning a request's entire prompt must fit within the available compute budget before prefill can execute.

We are not implementing chunked prefill in this section.

The scheduler is a metadata-level simulation. It models token computation and KV allocation without executing a real transformer or storing actual K/V tensors.

---

## 1. Defining the Request State

We begin by defining a `SimRequest` dataclass.

```python
from dataclasses import dataclass

@dataclass
class SimRequest:
    request_id: str
    arrival_step: int
    prompt_tokens: int
    max_new_tokens: int

    prefilled: bool = False
    generated: int = 0
    state: str = "NOT_ARRIVED"
```

The `@dataclass` decorator automatically generates an initializer and other useful methods based on the declared attributes.

This lets us create request objects without writing a separate `__init__()` method.

For example:

```python
request = SimRequest(
    request_id="A",
    arrival_step=1,
    prompt_tokens=40,
    max_new_tokens=10
)
```

Let's understand the purpose of each field.

### `request_id`

```python
request_id: str
```

A unique identifier for each inference request.

The scheduler uses this ID to associate a request with its physical KV block allocations.

For example, Request A's block table might be:

```text
A: [0, 1, 2]
```

The same identifier is used by `RequestBlockManager` to track the request's allocated physical blocks.

### `arrival_step`

```python
arrival_step: int
```

Specifies the scheduling iteration during which the request becomes available.

For example:

```text
Request A arrives at step 1
Request B arrives at step 2
Request C arrives at step 3
```

This lets us simulate requests arriving at different times rather than submitting every request before execution begins.

Continuous batching is useful precisely because an inference engine can handle newly arriving requests while existing requests are already generating tokens.

### `prompt_tokens`

```python
prompt_tokens: int
```

Represents the number of input tokens in the request's prompt.

During prefill, the model processes the prompt and produces the corresponding K/V vectors for the prompt tokens.

The scheduler uses this value for two purposes:

1. Determining how much compute budget prefill requires.
2. Determining how many physical KV blocks must be allocated.

For example:

```text
Prompt length: 40 tokens
Block size: 16 tokens

Required KV blocks: ceil(40 / 16) = 3
```

### `max_new_tokens`

```python
max_new_tokens: int
```

Specifies the maximum number of output tokens the request should generate.

In this simulation, requests finish when they reach this limit.

We are not implementing EOS-based stopping because no actual language model is generating token IDs.

### `prefilled`

```python
prefilled: bool = False
```

Tracks whether the request's initial prompt processing has completed.

Before prefill:

```text
prefilled = False
```

After successful prefill:

```text
prefilled = True
```

We maintain this field for explicit request metadata, although our scheduling logic primarily determines execution eligibility using the request's `state`.

### `generated`

```python
generated: int = 0
```

Tracks the number of output tokens sampled by the simulated model.

This is different from the number of tokens represented in the KV cache.

For example, after processing a 40-token prompt, a model typically produces logits from which the first output token is sampled.

At that point:

```text
Prompt KV tokens: 40
Generated output tokens: 1
Total KV tokens: 40
```

The first sampled output token does not yet have its own KV entry.

Its K/V vectors are created when that token is processed in the next decode forward pass.

This distinction is important for correct KV memory accounting.

### `state`

```python
state: str = "NOT_ARRIVED"
```

Represents the request's lifecycle state.

Our scheduler supports:

| State | Meaning |
|---|---|
| `NOT_ARRIVED` | Request has not entered the scheduler yet |
| `WAITING` | Request has arrived but has not completed prefill |
| `RUNNING` | Request has completed prefill and is eligible for decode |
| `FINISHED` | Request has completed generation and released its KV blocks |

KV-blocked execution is recorded as a scheduling event rather than a separate persistent request state.

For example, a running request can be temporarily KV-blocked because no additional physical block is available.

It remains in the `RUNNING` state and can be retried during a later iteration.

---

## 2. Initializing the Block-Aware Scheduler

Next, we define our scheduler class.

```python
class BlockAwareScheduler:

    def __init__(self, pool, manager, token_budget, requests):
        self.pool = pool
        self.manager = manager
        self.token_budget = token_budget
        self.requests = requests

        self.step_num = 0
        self.history = []
```

Let's understand each attribute.

### `self.pool`

```python
self.pool = pool
```

References our existing `BlockPool`.

The physical block pool maintains:

```python
free_blocks
allocated_blocks
```

It can tell us how many physical blocks remain available.

For example:

```text
Total blocks: 8
Allocated blocks: 6
Free blocks: 2
```

### `self.manager`

```python
self.manager = manager
```

References our existing `RequestBlockManager`.

The manager is responsible for:

```python
add_request()
grow_request()
locate_token()
release_request()
```

The scheduler uses these methods instead of manipulating the physical block tables directly.

This keeps the responsibilities separated.

The scheduler determines **which requests should execute**.

The memory manager determines **whether sufficient physical KV storage can be allocated**.

### `self.token_budget`

```python
self.token_budget = token_budget
```

Represents the maximum number of token positions that can be processed during one scheduling iteration.

In our experiment:

```python
token_budget = 50
```

A scheduling iteration can therefore process at most 50 tokens across all selected requests.

For example:

```text
Request A decode: 1 token
Request B prefill: 43 tokens

Total scheduled computation: 44 tokens
Remaining budget: 6 tokens
```

This is a simplified compute-budget model.

Actual execution time depends on many factors, including sequence lengths, batch size, hardware, attention kernels, and model architecture.

A token budget is useful because it places a limit on scheduled work without requiring us to estimate exact execution times.

### `self.requests`

```python
self.requests = requests
```

Stores the request objects managed by the scheduler.

Each request contains its arrival time, prompt length, generation limit, and current lifecycle state.

### `self.step_num`

```python
self.step_num = 0
```

Tracks the current scheduling iteration.

It begins at zero and increments each time `step()` executes.

### `self.history`

```python
self.history = []
```

Stores snapshots of the scheduler's behavior after each iteration.

These snapshots include scheduled operations, memory utilization, block tables, and execution events.

We'll use this history to analyze memory pressure and scheduling behavior in Part 7A.6.

### Input Validation

We also include:

```python
assert token_budget > 0
```

The compute budget must be positive.

```python
assert len(set(r.request_id for r in requests)) == len(requests)
```

This ensures that no two requests have the same identifier.

The set removes duplicate IDs. If its length differs from the number of requests, duplicate IDs exist.

Finally:

```python
assert all(
    r.prompt_tokens > 0
    and r.max_new_tokens > 0
    and r.arrival_step >= 1
    for r in requests
)
```

This ensures that every request has a positive prompt length, a positive generation limit, and a valid arrival step.

These checks prevent invalid workloads from entering our scheduler.

---

## 3. Finishing a Request and Releasing KV Memory

We implemented:

```python
def finish_request(self, request, events):
    rid = request.request_id

    freed = self.manager.block_tables[rid].copy()
    self.manager.release_request(rid)

    request.state = "FINISHED"
    events.append(f"{rid}: FINISHED, freed blocks {freed}")
```

This method is called when a request reaches its generation limit.

Let's understand each operation.

### Retrieve the request ID

```python
rid = request.request_id
```

We use a short local variable so that we don't have to repeatedly access `request.request_id`.

### Copy the allocated block IDs

```python
freed = self.manager.block_tables[rid].copy()
```

This retrieves the request's physical block table.

For example:

```text
Request A: [0, 1, 2, 6]
```

We use `.copy()` to preserve a snapshot of the block IDs before releasing the request.

This is useful for logging.

### Release the physical blocks

```python
self.manager.release_request(rid)
```

This delegates memory reclamation to `RequestBlockManager`.

Internally, the manager:

1. Retrieves the request's physical block IDs.
2. Returns those blocks to `BlockPool`.
3. Deletes the request's block table.
4. Deletes the request's KV token count.

The physical blocks become available for other requests.

### Update the request state

```python
request.state = "FINISHED"
```

The request is no longer eligible for scheduling.

### Record the completion event

```python
events.append(f"{rid}: FINISHED, freed blocks {freed}")
```

This records which physical blocks were released.

For example:

```text
A: FINISHED, freed blocks [0, 1, 2, 6]
```

This helps us observe when physical KV capacity becomes available again.

---

## 4. Validating Memory Accounting

Before implementing scheduling decisions, we need a way to verify that our block allocations remain consistent.

We introduced:

```python
def validate(self):
```

This method checks several invariants after each scheduler iteration.

### Collect all allocated physical blocks

```python
owned = [
    b
    for table in self.manager.block_tables.values()
    for b in table
]
```

This is a nested list comprehension.

Suppose our request block tables are:

```python
{
    "A": [0, 1, 2],
    "B": [3, 4, 5]
}
```

The expression produces:

```python
owned = [0, 1, 2, 3, 4, 5]
```

It collects every physical block ID currently assigned to a request.

### Check for duplicate ownership

```python
assert len(owned) == len(set(owned))
```

The set removes duplicate physical IDs.

If the lengths differ, at least one physical block is assigned to multiple requests.

Our current implementation assumes exclusive block ownership, so this must never happen.

In later prefix-caching implementations, block sharing can be intentional and requires reference counting.

### Verify allocator and manager consistency

```python
assert set(owned) == self.pool.allocated_blocks
```

Every physical block marked as allocated by `BlockPool` must belong to a request.

Similarly, every physical block referenced by a request must be marked as allocated in the pool.

This prevents inconsistencies such as:

```text
BlockPool says block 4 is allocated.

But no request owns block 4.
```

Such a mismatch could indicate a memory leak or an incorrect allocation update.

### Verify total block accounting

```python
assert (
    self.pool.num_free_blocks
    + self.pool.num_allocated_blocks
    == self.pool.num_blocks
)
```

Every physical block must be either free or allocated.

For example:

```text
Total blocks: 8

Allocated: 6
Free: 2

6 + 2 = 8
```

### Ensure free and allocated blocks do not overlap

```python
assert set(self.pool.free_blocks).isdisjoint(
    self.pool.allocated_blocks
)
```

`isdisjoint()` returns `True` when two sets have no elements in common.

A physical block cannot be simultaneously available for allocation and owned by an active request.

### Ensure the free list has no duplicates

```python
assert len(self.pool.free_blocks) == len(
    set(self.pool.free_blocks)
)
```

If the same physical block ID appeared twice in the free list, the allocator might assign it to two different requests.

This check prevents that situation.

### Validate running requests

For every request, we inspect its current lifecycle state.

```python
for r in self.requests:
    rid = r.request_id
```

If the request is running:

```python
if r.state == "RUNNING":
    assert rid in self.manager.block_tables
```

Every running request must have an active block table.

Next:

```python
expected_kv = (
    r.prompt_tokens + r.generated - 1
)
```

This is one of the most important calculations in this notebook.

Why do we subtract one?

Because the most recently sampled output token has not yet been processed into KV.

Suppose Request A has:

```text
Prompt tokens: 40
Generated output tokens: 5
```

Its KV cache contains:

```text
40 prompt tokens
4 processed output tokens
```

Therefore:

\[
\text{KV Tokens} = 40 + 5 - 1 = 44
\]

The fifth output token has been sampled but will only receive its KV entry during the next decode forward pass.

We verify that the memory manager agrees:

```python
assert self.manager.token_counts[rid] == expected_kv
```

Then we calculate how many physical blocks this KV length requires:

```python
expected_blocks = self.manager.blocks_needed(
    expected_kv
)
```

And verify:

```python
assert (
    len(self.manager.block_tables[rid])
    == expected_blocks
)
```

This ensures that the request has the correct number of allocated physical blocks for its recorded KV token count.

For requests that are not running:

```python
else:
    assert rid not in self.manager.block_tables
```

A request that has not arrived, is waiting for prefill, or has finished must not own physical KV blocks in our implementation.

This is consistent with our all-or-nothing prefill and immediate-release policies.

---

## 5. Executing a Scheduler Iteration

The central method of our implementation is:

```python
def step(self):
```

One call to `step()` represents one scheduling iteration.

During this iteration, the scheduler:

1. Registers newly arriving requests.
2. Schedules decoding for existing running requests.
3. Attempts prefill for waiting requests.
4. Verifies memory accounting.
5. Records a snapshot of the iteration.
6. Prints scheduling decisions and memory state.

Let's examine each stage.

### Initialize the iteration

```python
self.step_num += 1
budget = self.token_budget
events = []
scheduled = []
```

`self.step_num += 1` advances the scheduling iteration counter.

`budget` is a local variable representing the remaining compute budget for the current iteration.

For our experiment:

```text
Initial budget: 50 tokens
```

The budget resets at the beginning of every scheduling iteration.

`events` stores descriptive messages about arrivals, allocation failures, prefill, decoding, and request completion.

`scheduled` stores the actual simulated computation operations selected for the iteration.

For example:

```python
scheduled = [
    ("A", "DECODE", 1),
    ("B", "PREFILL", 43)
]
```

This indicates that Request A processed one decode token while Request B processed its 43-token prompt.

The total compute budget consumed is 44 tokens.

---

## 6. Handling New Request Arrivals

The first phase of `step()` handles incoming requests.

```python
for r in self.requests:
    if (
        r.state == "NOT_ARRIVED"
        and r.arrival_step <= self.step_num
    ):
        r.state = "WAITING"
```

We iterate over every request.

A request becomes eligible for scheduling when:

1. It has not arrived previously.
2. Its configured arrival step has been reached.

For example:

```text
Current scheduler step: 2

Request A arrival step: 1
Request B arrival step: 2
Request C arrival step: 3
```

At step 2, Request B becomes available.

Request C remains in the `NOT_ARRIVED` state.

Once a request arrives:

```python
r.state = "WAITING"
```

It enters the waiting state.

This does not mean its prompt has been processed.

It simply means the scheduler can now consider it for prefill.

We also record:

```python
events.append(
    f"{r.request_id}: ARRIVED "
    f"(prompt={r.prompt_tokens}, "
    f"max_new={r.max_new_tokens})"
)
```

For example:

```text
B: ARRIVED (prompt=43, max_new=7)
```

This gives us visibility into when requests enter the scheduling system.

---

## 7. Decode-First Scheduling

After handling arrivals, we schedule existing running requests.

```python
for r in self.requests:

    if r.state != "RUNNING":
        continue
```

The scheduler iterates through all requests.

If a request is not running, it is skipped using `continue`.

Only requests that have already completed prefill are eligible for decoding.

### Check available compute budget

```python
if budget < 1:
    events.append(
        f"{r.request_id}: decode deferred, "
        "compute budget exhausted"
    )
    continue
```

Each decode operation in our simulation processes one token position.

Therefore, it requires one unit of the compute-token budget.

If no budget remains, decoding is deferred until a later iteration.

### Retrieve the request's current KV length

```python
rid = r.request_id

current_kv = self.manager.token_counts[rid]
```

The request manager tells us how many token positions currently have KV storage accounted for.

For example:

```text
Request A

Prompt tokens: 40
Generated tokens: 1
Current KV tokens: 40
```

### Calculate the next KV length

```python
target_kv = current_kv + 1
```

A decode forward pass processes the most recently sampled output token.

That processing creates one additional K/V entry per transformer layer.

Therefore, the stored KV length increases by one.

For example:

```text
Before decode: 40 KV tokens
After decode: 41 KV tokens
```

### Request additional KV capacity if needed

```python
success = self.manager.grow_request(
    rid, target_kv
)
```

This calls the `grow_request()` method implemented in Section 7A.4.

The manager calculates whether the new KV length fits in the existing allocated blocks.

For example:

```text
Current KV tokens: 40
Target KV tokens: 41

Allocated blocks: 3
Capacity: 48 tokens

Additional blocks required: 0
```

No new physical block is needed.

But at a block boundary:

```text
Current KV tokens: 48
Target KV tokens: 49

Allocated blocks: 3
Capacity: 48 tokens

Additional blocks required: 1
```

Now the manager must acquire another physical block.

This is where physical KV memory availability directly influences scheduling.

### Handle KV allocation failure

```python
if not success:
    events.append(
        f"{rid}: KV-BLOCKED during decode "
        f"(free_blocks={self.pool.num_free_blocks})"
    )
    continue
```

If the manager cannot allocate the required physical block, the decode operation is deferred.

The request remains in the running state.

No compute budget is consumed because the simulated forward pass does not execute.

The request's generated-token count and committed KV length remain unchanged.

This is important because allocation failure must not produce partially committed model execution.

### Simulate successful decoding

If the KV allocation succeeds:

```python
budget -= 1
r.generated += 1
scheduled.append((rid, "DECODE", 1))
```

`budget -= 1` consumes one unit of the compute budget.

`r.generated += 1` represents sampling one additional output token after the decode forward pass.

`scheduled.append(...)` records the operation.

For example:

```text
A: DECODE (1 computed token)
```

Remember the distinction:

The decode forward pass processes the previous output token into the KV cache and produces logits for sampling the next output token.

Therefore, both the committed KV length and generated-output count increase by one.

We record the event:

```python
events.append(
    f"{rid}: DECODE, "
    f"generated={r.generated}/{r.max_new_tokens}, "
    f"KV_tokens={target_kv}"
)
```

### Finish requests that reach their generation limit

```python
if r.generated >= r.max_new_tokens:
    self.finish_request(r, events)
```

Once a request reaches its configured maximum output length, it finishes.

Its physical KV blocks are released immediately.

This means blocks reclaimed by a completed decode request can potentially be reused by waiting requests during the prefill phase of the same scheduling iteration.

---

## 8. Prefill Scheduling

After scheduling decode operations, the scheduler attempts to admit waiting requests.

```python
for r in self.requests:

    if r.state != "WAITING":
        continue
```

Only requests in the waiting state are considered for prefill.

### Check whether the prompt fits the remaining compute budget

```python
if r.prompt_tokens > budget:
    events.append(
        f"{rid}: PREFILL DEFERRED "
        f"(needs={r.prompt_tokens}, budget={budget})"
    )
    break
```

This is the all-or-nothing prefill policy from our earlier token-budget-aware scheduler.

Suppose:

```text
Remaining compute budget: 30 tokens
Request prompt length: 43 tokens
```

The request cannot be prefetched during this iteration.

We do not process the first 30 tokens and defer the remaining 13.

That behavior would require chunked prefill, which we have not implemented yet.

Instead, the entire prefill operation is deferred.

### Why do we use `break`?

`break` stops the waiting-request loop.

This preserves the admission ordering used in our FIFO scheduler.

For example:

```text
Waiting queue:

B: 43 tokens
C: 20 tokens

Remaining compute budget: 30
```

Request B cannot fit.

Even though Request C could fit, our scheduler does not skip B to admit C.

It stops processing the waiting queue for the current iteration.

This maintains FIFO ordering but may leave some compute budget unused.

Different production schedulers can use different policies. Our implementation deliberately retains the policy introduced in previous parts.

### Allocate physical KV blocks for prefill

If the prompt fits the remaining compute budget, we attempt to allocate its KV blocks.

```python
success = self.manager.add_request(
    rid, r.prompt_tokens
)
```

For example, a 43-token prompt needs:

\[
\left\lceil\frac{43}{16}\right\rceil = 3 \text{ blocks}
\]

The request manager asks `BlockPool` for three physical block IDs.

If enough blocks are available, the request is admitted.

Otherwise, allocation fails.

### Handle insufficient physical KV capacity

```python
if not success:
    events.append(
        f"{rid}: KV-BLOCKED during prefill "
        f"(needs_blocks="
        f"{self.manager.blocks_needed(r.prompt_tokens)}, "
        f"free_blocks={self.pool.num_free_blocks})"
    )
    break
```

This is the main difference between our earlier token-capacity scheduler and the current block-aware implementation.

Previously, we reasoned about KV capacity primarily in token slots.

Now, the scheduler must satisfy physical block allocation requirements.

For example:

```text
Available physical blocks: 2
Request C needs: 3 blocks

Allocation result: Failed
```

Even if enough compute budget remains, the request cannot execute prefill because insufficient physical KV storage is available.

The request remains waiting.

### Simulate successful prefill

If allocation succeeds:

```python
budget -= r.prompt_tokens
r.prefilled = True
r.state = "RUNNING"
```

The prompt length is deducted from the compute budget.

The request is marked as prefilled and transitions into the running state.

Next:

```python
r.generated = 1
```

Why do we set generated tokens to one?

Because prefill processes the prompt and produces logits from which the first output token can be sampled.

In an autoregressive decoder-only model, the prompt's final position can produce the first next-token prediction.

Therefore:

```text
Prompt tokens: 43
Committed KV tokens: 43
Generated output tokens: 1
```

The sampled output token has not yet been processed into KV.

This is consistent with our earlier memory-accounting formula:

\[
\text{KV Tokens} =
\text{Prompt Tokens} + \text{Generated Tokens} - 1
\]

For example:

\[
43 + 1 - 1 = 43
\]

We record the scheduled operation:

```python
scheduled.append(
    (rid, "PREFILL", r.prompt_tokens)
)
```

And log the completion:

```python
events.append(
    f"{rid}: PREFILL completed "
    f"({r.prompt_tokens} tokens), "
    f"sampled first output token"
)
```

### Handle requests requiring only one output token

```python
if r.generated >= r.max_new_tokens:
    self.finish_request(r, events)
```

If the request only needs one generated output token, it can finish immediately after prefill.

No additional decode iteration is required.

Its KV blocks are released.

---

## 9. Validating and Recording Each Scheduling Step

After decode and prefill scheduling, we perform our memory consistency checks.

```python
self.validate()
```

This verifies that the scheduler and request manager agree on physical block ownership and KV token counts.

Next, we create a snapshot:

```python
snapshot = {
    "step": self.step_num,
    "scheduled": scheduled.copy(),
    "events": events.copy(),
    "budget_used": self.token_budget - budget,
    "free_blocks": self.pool.num_free_blocks,
    "allocated_blocks": self.pool.num_allocated_blocks,
    "block_tables": {
        rid: table.copy()
        for rid, table in self.manager.block_tables.items()
    }
}
```

Let's understand each entry.

**`step`**

The current scheduler iteration number.

**`scheduled`**

Records the simulated compute operations selected during the iteration.

For example:

```python
[
    ("A", "DECODE", 1),
    ("B", "PREFILL", 43)
]
```

**`events`**

Stores descriptive messages about arrivals, scheduling decisions, allocation failures, and request completion.

**`budget_used`**

Calculates how much compute budget was consumed:

```python
self.token_budget - budget
```

For example:

```text
Initial budget: 50
Remaining budget: 6

Budget used: 44
```

**`free_blocks`**

Records the number of free physical blocks at the end of the iteration.

**`allocated_blocks`**

Records the number of allocated physical blocks at the end of the iteration.

**`block_tables`**

Creates a snapshot of every active request's physical block table.

We use `.copy()` so that future changes to block tables do not overwrite the recorded snapshot.

For example:

```python
{
    "A": [0, 1, 2],
    "B": [3, 4, 5]
}
```

Finally:

```python
self.history.append(snapshot)
```

The snapshot is added to the scheduler's execution history.

This will be useful for analyzing memory allocation and scheduling behavior in the next section.

### Printing the Step Summary

The remaining lines in `step()` print:

- Events that occurred.
- Scheduled prefill and decode operations.
- Compute budget consumption.
- Allocated and free KV blocks.
- Current request block tables.
- Request lifecycle states and generated-token counts.

These print statements do not modify scheduling decisions.

They simply expose the internal state so we can observe how the scheduler behaves.

---

## 10. Running the Scheduler Until Completion

We implemented:

```python
def run(self, max_steps=100):
```

This method repeatedly invokes the scheduler until all requests finish or the configured iteration limit is reached.

### Execute scheduling iterations

```python
for _ in range(max_steps):
```

This loop allows up to `max_steps` scheduling iterations.

We use an explicit limit to prevent an infinite loop if requests become permanently blocked.

### Check whether all requests are finished

```python
if all(
    r.state == "FINISHED"
    for r in self.requests
):
    break
```

The `all()` function returns `True` only if every request satisfies the condition.

Once all requests have finished, the loop stops.

Otherwise:

```python
self.step()
```

The scheduler executes another iteration.

### Detect incomplete simulations

After the loop:

```python
if not all(
    r.state == "FINISHED"
    for r in self.requests
):
    raise RuntimeError(
        "Simulation did not complete. "
        "Check compute budget, KV capacity, "
        "or blocked requests."
    )
```

This catches situations where the scheduler cannot finish the workload within the configured iteration limit.

For example, a request might require more KV blocks than the total pool contains.

Under our non-preemptive, all-or-nothing allocation policy, such a request could remain blocked indefinitely.

Rather than silently reporting success, the simulation raises an error.

### Perform final memory checks

```python
self.validate()

assert self.pool.num_allocated_blocks == 0
assert len(self.manager.block_tables) == 0
```

Once all requests finish, no active request should retain physical KV blocks.

The first assertion verifies that the physical pool has no allocated blocks.

The second verifies that the request manager has no remaining block tables.

These checks help detect allocations that were not released.

Finally, the method prints the total scheduling steps, available physical blocks, and the completion status of each request.

---

## 11. Initializing Physical KV Memory

We used the same block size as the previous sections.

```python
BLOCK_SIZE = 16
```

Each physical KV block has capacity for 16 token positions.

Next:

```python
pool = BlockPool(num_blocks=8)
```

We created a pool containing 8 physical blocks.

The total capacity is:

\[
8 \times 16 = 128 \text{ token slots}
\]

For Qwen2.5-0.5B using FP16 KV storage, each block represents 192 KiB of KV memory.

Therefore:

\[
8 \times 192 = 1536 \text{ KiB}
\]

Our physical block pool represents 1.5 MiB of KV storage capacity.

Next:

```python
manager = RequestBlockManager(
    pool=pool,
    block_size=BLOCK_SIZE
)
```

This creates a request-aware block manager using the same physical block pool.

Every request will compete for capacity from these 8 blocks.

The small pool makes it easier to observe how allocation pressure affects scheduling decisions.

---

## 12. Defining the Request Workload

We created three simulated inference requests.

```python
requests = [
    SimRequest(
        request_id="A",
        arrival_step=1,
        prompt_tokens=40,
        max_new_tokens=10
    ),
    SimRequest(
        request_id="B",
        arrival_step=2,
        prompt_tokens=43,
        max_new_tokens=7
    ),
    SimRequest(
        request_id="C",
        arrival_step=3,
        prompt_tokens=38,
        max_new_tokens=4
    )
]
```

Their configurations are:

| Request | Arrival step | Prompt tokens | Maximum output tokens |
|---|---:|---:|---:|
| A | 1 | 40 | 10 |
| B | 2 | 43 | 7 |
| C | 3 | 38 | 4 |

The prompts have different lengths and the requests arrive at different times.

This lets us observe continuous batching and KV memory competition.

The initial KV block requirements are:

```text
A: ceil(40 / 16) = 3 blocks
B: ceil(43 / 16) = 3 blocks
C: ceil(38 / 16) = 3 blocks
```

Together, all three requests initially require 9 blocks.

However, our pool contains only 8.

Therefore, all three requests cannot hold their complete prompt KV allocations simultaneously.

At least one request must wait for available physical capacity.

This is intentional.

We want the experiment to demonstrate a real memory-management constraint rather than letting every request execute without contention.

---

## 13. Creating and Running the Scheduler

We initialized the scheduler:

```python
scheduler = BlockAwareScheduler(
    pool=pool,
    manager=manager,
    token_budget=50,
    requests=requests
)
```

The scheduler receives:

- The physical block pool.
- The request block manager.
- A compute-token budget of 50.
- The three simulated requests.

Finally:

```python
scheduler.run(max_steps=100)
```

This begins the scheduling simulation.

The scheduler repeatedly processes arrivals, selects decode and prefill operations, allocates KV blocks, and releases completed requests.

Let's examine how our chosen workload behaves.

---

## 14. Understanding the Scheduler Execution

### Step 1: Request A Arrives

Request A arrives with a 40-token prompt.

Available compute budget:

```text
50 tokens
```

Available physical KV blocks:

```text
8 blocks
```

Request A requires:

```text
40 compute tokens
3 physical KV blocks
```

Both requirements can be satisfied.

The scheduler performs prefill and allocates physical blocks `[0, 1, 2]`.

After prefill:

```text
Request A

State: RUNNING
Generated tokens: 1
Committed KV tokens: 40
Physical blocks: [0, 1, 2]
```

The first output token is sampled from the prefill logits.

The scheduler consumes 40 compute tokens.

### Step 2: Request A Decodes, Request B Arrives

At the beginning of Step 2, Request A is already running.

Request B arrives with a 43-token prompt.

Our scheduler prioritizes decoding before admitting waiting requests.

First, Request A decodes one token.

Its KV length grows from 40 to 41.

No additional physical block is required because its existing three blocks provide capacity for 48 tokens.

The compute budget decreases from 50 to 49.

Next, the scheduler considers Request B.

Request B requires:

```text
43 compute tokens
3 physical KV blocks
```

Both requirements can be satisfied.

The scheduler admits B and allocates blocks `[3, 4, 5]`.

After Step 2:

```text
Request A:
  Generated: 2
  KV tokens: 41
  Blocks: [0, 1, 2]

Request B:
  Generated: 1
  KV tokens: 43
  Blocks: [3, 4, 5]

Free physical blocks: 2
Compute budget used: 44
```

This demonstrates continuous batching.

Request B enters the active batch while Request A is already decoding.

### Step 3: Request C Arrives but Cannot Be Admitted

Request C arrives with a 38-token prompt.

At this point:

```text
Request A: 3 blocks
Request B: 3 blocks

Total allocated: 6 blocks
Free blocks: 2
```

The scheduler first processes one decode token for A and one for B.

Neither request crosses a block boundary yet.

Request C then attempts prefill.

Its prompt requires 3 physical blocks.

However, only 2 remain available.

The allocation fails:

```text
C: KV-BLOCKED during prefill
```

Request C remains waiting.

The remaining compute budget cannot help because the limiting resource is physical KV cache capacity.

This illustrates why **compute availability and KV memory availability are separate scheduling constraints**.

A request can have sufficient compute budget but still be unable to execute because its KV allocation cannot be satisfied.

### Steps 4 to 6: Existing Requests Continue Decoding

Requests A and B continue decoding.

Their KV lengths grow incrementally.

Request A approaches the 48-token capacity of its three blocks.

Request B also approaches its block boundary.

As long as the new KV length fits inside the existing block allocation, `grow_request()` updates the token count without requesting additional physical blocks.

Request C remains blocked because the pool still contains only 2 free blocks.

### Step 7: Request B Finishes

At Step 7, Request B reaches its generation limit of 7 output tokens.

Its final decode processes the previously sampled token, produces the seventh output token, and then finishes.

The scheduler calls:

```python
self.finish_request(r, events)
```

Request B's physical blocks are released.

The physical block pool now has enough free capacity for Request C.

During the same scheduling iteration, the scheduler proceeds to the prefill phase.

Request C is admitted and receives three physical blocks.

This demonstrates **KV memory reclamation and reuse during continuous batching**.

As soon as one request finishes, its released blocks can be assigned to another waiting request.

### Step 8: Request A Continues and C Begins Decoding

Request A continues decoding.

Request C has completed prefill and can now perform decode operations.

Both requests can be scheduled within the same compute-token budget, provided sufficient KV capacity exists.

Request A's KV allocation may need an additional physical block when its length crosses 48 tokens.

The request manager handles this through `grow_request()`.

### Step 10: Request A Finishes

Request A reaches its configured generation limit of 10 output tokens.

Its physical KV blocks are released.

Request C continues using its own allocated KV blocks.

This demonstrates that requests can finish independently without affecting other active requests.

### Step 11: Request C Finishes

Request C reaches its generation limit of 4 output tokens.

Its allocated KV blocks are released.

No active requests remain.

The scheduler terminates.

---

## 15. Final Simulation Result

Our workload completes in 11 scheduling iterations.

The expected final summary is:

```text
SIMULATION COMPLETE

Total scheduling steps: 11

Free physical KV blocks: 8
Allocated physical KV blocks: 0

Request A: 10 output tokens, FINISHED
Request B: 7 output tokens, FINISHED
Request C: 4 output tokens, FINISHED
```

All three requests completed within their configured generation limits.

All physical KV blocks were returned to the free pool.

The request manager no longer maintains block tables for finished requests.

This confirms that our scheduler can coordinate request execution, block allocation, and memory reclamation for this workload.

---

## 16. Why This Is Different From Our Previous KV-Capacity Scheduler

In Part 6, our scheduler reasoned about KV memory capacity using token-level accounting.

For example:

```text
Available KV capacity: 88 token slots
```

It compared the number of required KV token positions against the remaining capacity.

In this section, we replaced that abstract capacity model with **physical block-level allocation**.

For example:

```text
Total physical blocks: 8
Block size: 16 tokens

Total capacity: 128 token slots
```

A request containing 40 tokens still consumes 3 complete physical blocks.

Therefore:

```text
Actual KV positions: 40
Allocated capacity: 48 token slots
Unused slots: 8
```

The scheduler must account for those 8 unused slots because they belong to Request A's allocated physical blocks.

Another request cannot use them under our exclusive block-ownership model.

This introduces a more realistic constraint compared with token-level capacity accounting.

The scheduler now operates on the number of physical blocks available rather than assuming that every unused token slot can be shared freely across requests.

This is especially important when sequences have different lengths and grow dynamically.

---

## 17. Important Implementation Limitations

Our scheduler demonstrates several real inference-serving concepts, but it is not a complete production inference engine.

### No actual transformer execution

We simulate successful prefill and decode operations.

We do not run a model forward pass or generate real token IDs.

The scheduler updates token counts as if the corresponding model computation succeeded.

### No actual GPU KV tensors

The physical block pool manages block IDs and metadata.

It does not allocate or write Key and Value tensors.

In Part 7B, we'll connect block mappings to actual paged K/V storage and attention computations.

### No chunked prefill

Prefill is all-or-nothing.

A prompt must fit entirely within the remaining compute-token budget.

If it does not fit, the request is deferred.

Chunked prefill would allow large prompts to be processed across multiple scheduling iterations.

We have intentionally left that for a separate implementation.

### No preemption

When a request cannot obtain an additional KV block, we defer its execution.

We do not preempt another request or recompute its KV cache.

Production schedulers may use preemption to make progress under memory pressure.

### No prefix caching or shared physical blocks

Every allocated physical block belongs to exactly one request.

We do not reuse cached KV data across requests with shared prompt prefixes.

This keeps memory ownership simple and allows our validator to reject duplicate block ownership.

### Simplified scheduling policy

Our scheduler processes running decode requests before waiting prefill requests.

It also preserves FIFO ordering for waiting requests.

These are deliberate policy choices for this notebook, not universal requirements of every inference engine.

Real serving engines may use more sophisticated scheduling policies depending on their goals and supported features.

---

## What We've Built

We now have a block-aware continuous batching scheduler that integrates the memory-management components implemented throughout Part 7A.

It supports:

1. Dynamic request arrivals.
2. Waiting, running, and finished request states.
3. Decode-first scheduling.
4. All-or-nothing prefill.
5. Per-iteration token-budget enforcement.
6. Per-request physical KV block tables.
7. Incremental KV allocation during decoding.
8. Deferring execution when KV capacity is insufficient.
9. Releasing physical blocks after request completion.
10. Reusing released blocks for newly admitted requests.
11. Recording scheduler history and block allocation state.
12. Validating KV memory ownership and token accounting after each iteration.

The key idea is that **the scheduler must coordinate both computation and physical KV memory allocation**.

Having enough compute budget does not guarantee that a request can execute.

The request must also have enough physical KV block capacity to store the K/V data that its next forward pass will produce.

By integrating `BlockPool` and `RequestBlockManager` with the scheduler, we've moved from abstract KV capacity accounting toward a block-based memory-management system inspired by real LLM inference engines.
